# RAG Evaluation

This notebook evaluates the Clinical Trial Protocol Intelligence Copilot using questions that were not used to tune the retrieval or generation pipeline.

The goal is to measure retrieval, answer grounding, citation behaviour, and abstention separately.

The earlier Day 3 questions are treated as development smoke tests rather than final evaluation data. Day 5 will create a new manually verified evaluation set so that improvements are based on observed failures rather than repeated tuning against familiar questions.

## 1. Define the evaluation framework

A RAG system can fail in different ways.

Retrieval may fail to find the correct evidence. The language model may receive the correct evidence but still produce an unsupported answer. Citations may point to the wrong evidence, or the model may answer a question that should have been rejected.

We will therefore evaluate retrieval, answer correctness, citation grounding, and abstention separately instead of reporting one overall accuracy number.

## 2. Design the held-out evaluation set

The final evaluation will target 40 new questions across the five protocols.

Each protocol will contribute six supported questions and two deliberately unsupported questions. Supported questions will include straightforward facts, paraphrased or difficult questions, potentially confusable information, and structure-heavy evidence where appropriate.

The questions will be manually verified against the source chunks before being used for evaluation. Questions previously used during Day 3 development will not contribute to the final held-out metrics.

In [5]:
import pandas as pd
import re


# Define the five protocols in the evaluation corpus
protocols = [
    ("PROTO_001", "CERTAIN"),
    ("PROTO_002", "CARE_STROKE"),
    ("PROTO_003", "INTEGRA"),
    ("PROTO_004", "LISTEN"),
    ("PROTO_005", "THP_TA"),
]


# Plan eight evaluation questions for each protocol
question_plan = [
    ("straightforward", "supported"),
    ("straightforward", "supported"),
    ("difficult", "supported"),
    ("difficult", "supported"),
    ("confusable", "supported"),
    ("structure_heavy", "supported"),
    ("unsupported", "unsupported"),
    ("unsupported", "unsupported"),
]


evaluation_rows = []

question_number = 1


# Create the evaluation blueprint
for document_id, document_name in protocols:

    for question_type, expected_support in question_plan:

        evaluation_rows.append(
            {
                "question_id": f"EVAL_{question_number:03d}",
                "document_id": document_id,
                "document_name": document_name,
                "question_type": question_type,
                "expected_support": expected_support,
                "question": "",
                "gold_chunk_ids": "",
                "gold_pages": "",
                "gold_answer_notes": "",
                "verification_notes": "",
            }
        )

        question_number += 1


evaluation_df = pd.DataFrame(
    evaluation_rows
)


print("Evaluation questions planned:", len(evaluation_df))
print()
print(
    evaluation_df.groupby(
        ["document_name", "expected_support"]
    )
    .size()
)

Evaluation questions planned: 40

document_name  expected_support
CARE_STROKE    supported           6
               unsupported         2
CERTAIN        supported           6
               unsupported         2
INTEGRA        supported           6
               unsupported         2
LISTEN         supported           6
               unsupported         2
THP_TA         supported           6
               unsupported         2
dtype: int64


## 3. Load the validated protocol chunks

The evaluation questions must be created from verified source evidence rather than from memory or from the RAG system's own answers.

We will load the validated chunks created on Day 2 so that each supported evaluation question can be linked to the exact document, page, and chunk containing the correct evidence.

In [2]:
# Load the validated chunks created during Day 2
chunks_df = pd.read_json(
    "../data/processed/protocol_chunks.jsonl",
    lines=True,
)


# Check that the complete chunk corpus loaded correctly
print("Chunks loaded:", len(chunks_df))
print("Documents:", chunks_df["document_id"].nunique())
print("Pages represented:", chunks_df[["document_id", "page_number"]].drop_duplicates().shape[0])

print("\nChunks per protocol:")
print(
    chunks_df.groupby("document_name")
    .size()
)

Chunks loaded: 172
Documents: 5
Pages represented: 52

Chunks per protocol:
document_name
CARE_STROKE    23
CERTAIN        28
INTEGRA        39
LISTEN         39
THP_TA         43
dtype: int64


## 4. Find candidate evidence in CERTAIN

We will begin building the held-out evaluation set using the CERTAIN protocol.

Before writing questions, we will inspect the actual source chunks for useful topics. Questions used during Day 3 development, such as the CERTAIN primary outcome and sample size questions, will not be reused in the final evaluation metrics.

The goal is to identify several different types of evidence from which new questions can be manually created and verified.

In [3]:
# Keep only CERTAIN chunks
certain_chunks = (
    chunks_df[
        chunks_df["document_name"] == "CERTAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


# Search for useful protocol topics
candidate_terms = [
    "inclusion",
    "exclusion",
    "eligib",
    "secondary outcome",
    "random",
    "follow",
    "intervention",
    "baseline",
    "blind",
    "adverse",
]


# Find chunks containing each candidate term
for term in candidate_terms:

    matches = certain_chunks[
        certain_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
        )
    ]

    print(f"\nTERM: {term}")
    print("Matching chunks:", len(matches))

    # Show only a few useful matches for inspection
    for _, row in matches.head(3).iterrows():

        print(
            f"\n{row['chunk_id']} | Page {row['page_number']}"
        )

        # Print a manageable text preview
        print(
            row["text"][:900]
            .replace("\n", " ")
        )


TERM: inclusion
Matching chunks: 3

PROTO_001_P001_C003 | Page 1
CTRI/2019/05/019484. INTRODUCTION Tobacco use is responsible for almost eight  million deaths each year or a death every 6  seconds.1 2 About 80% of 1.3  billion tobacco  consumers across the globe live in low and  middle-  income  countries (LMICs).1 In LMICs  like India, the tobacco problem is complex as  the country has a diverse population with a  mixture of cultures, religions and practices.  The majority of the tobacco users in India use  a variety of tobacco products—combustible,  non-  combustible or both. As per the Global  Adult T obacco Survey 2 (GATS 2), 28.6%  of the adult population in India consumes  tobacco (10.7% of smoking and 21.4% of  smokeless), making it the second-  largest  consumer in the world.3 Widespread use of smokeless tobacco  (SLT) products occurs in countries such as in  Strengths and limitations of this study  ► Evidence on the effectiveness of smokeless

PROTO_001_P001_C004 | Page 1
the

## 5. Inspect CERTAIN evidence around candidate topics

The first evidence search produced useful candidate chunks, but the output was too large to inspect reliably.

We will now display short passages around each matching term. This makes it easier to identify exact facts that can become manually verified evaluation questions.

In [6]:
def show_term_context(
    dataframe,
    term,
    context_chars=350,
    max_matches=5,
):
    """Show a short passage around a search term."""

    matches = dataframe[
        dataframe["text"].str.contains(
            term,
            case=False,
            na=False,
        )
    ]

    print(f"TERM: {term}")
    print("Matching chunks:", len(matches))

    for _, row in matches.head(max_matches).iterrows():

        text = row["text"]

        # Find the first occurrence of the search term
        match = re.search(
            term,
            text,
            flags=re.IGNORECASE,
        )

        if match is None:
            continue

        # Keep a small amount of text before and after the match
        start = max(
            0,
            match.start() - context_chars,
        )

        end = min(
            len(text),
            match.end() + context_chars,
        )

        snippet = text[start:end].replace(
            "\n",
            " ",
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


# Inspect the most useful CERTAIN topics individually
for term in [
    "inclusion",
    "eligible",
    "randomisation",
    "follow",
    "intervention",
    "mask",
]:

    print()
    show_term_context(
        certain_chunks,
        term,
    )


TERM: inclusion
Matching chunks: 3

PROTO_001_P001_C003 | Page 1
.  ► The primary care physicians may get unmasked to  the allocation of the participants during the course  of the trial as the participants visiting the urban pri- mary health centres may reveal the information to  the physicians. There is a chance of biased results if  physicians may deliver repeated face-  to-  face counselling services.  ► The inclusion criteria include SLT users having a  mobile phone, this may exclude a section of the SLT  users who do not have a personal mobile phone like

PROTO_001_P001_C004 | Page 1
the physicians. There is a chance of biased results if  physicians may deliver repeated face-  to-  face counselling services.  ► The inclusion criteria include SLT users having a  mobile phone, this may exclude a section of the SLT  users who do not have a personal mobile phone like  very poor or the elderly who do not use a mobile  phone. Protected by copyright, including for uses related to text a

## 6. Identify the exact CERTAIN evidence chunks

Several useful evaluation topics were identified, but the previous output was too long to inspect completely.

We will now list only the matching chunk IDs and page numbers for the main candidate topics. This will help us select a small set of exact chunks for manual verification before creating the gold-standard questions.

In [7]:
# List only the chunk IDs and pages for useful CERTAIN topics
evidence_terms = [
    "inclusion",
    "randomisation",
    "follow",
    "intervention",
    "mask",
    "blinded",
]


for term in evidence_terms:

    matches = certain_chunks[
        certain_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
        )
    ]

    print(f"\nTERM: {term}")

    if matches.empty:
        print("No matching chunks")
        continue

    print(
        matches[
            [
                "chunk_id",
                "page_number",
            ]
        ].to_string(index=False)
    )


TERM: inclusion
           chunk_id  page_number
PROTO_001_P001_C003            1
PROTO_001_P001_C004            1
PROTO_001_P006_C002            6

TERM: randomisation
           chunk_id  page_number
PROTO_001_P001_C003            1
PROTO_001_P002_C003            2
PROTO_001_P002_C004            2
PROTO_001_P003_C002            3
PROTO_001_P004_C001            4
PROTO_001_P005_C001            5

TERM: follow
           chunk_id  page_number
PROTO_001_P001_C002            1
PROTO_001_P002_C001            2
PROTO_001_P002_C002            2
PROTO_001_P002_C003            2
PROTO_001_P002_C004            2
PROTO_001_P003_C001            3
PROTO_001_P003_C002            3
PROTO_001_P003_C003            3
PROTO_001_P004_C001            4
PROTO_001_P004_C003            4
PROTO_001_P005_C002            5
PROTO_001_P005_C003            5
PROTO_001_P005_C004            5
PROTO_001_P006_C002            6
PROTO_001_P006_C003            6
PROTO_001_P007_C005            7
PROTO_001_P007_C006     

## 7. Verify specific CERTAIN evidence

The broad topic search identified several useful areas of the CERTAIN protocol.

We will now inspect six specific facts that were not used in the Day 3 development questions. Each fact must be linked to exact source evidence before it can become part of the gold-standard evaluation set.

In [8]:
# Define specific phrases for six candidate CERTAIN facts
certain_evidence_searches = {
    "mobile_phone_requirement": "mobile phone",
    "randomisation_method": "randomisation will be done",
    "follow_up_duration": "follow-up is only for 3 months",
    "outcome_staff_blinding": "staff collecting outcome data will be blinded",
    "face_to_face_counselling": "single 10-",
    "routine_care_component": "routine care",
}


for topic, phrase in certain_evidence_searches.items():

    print(f"\nTOPIC: {topic}")

    matches = certain_chunks[
        certain_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print("Matching chunks:", len(matches))

    # Show at most two matches for each specific fact
    for _, row in matches.head(2).iterrows():

        text = row["text"]
        position = text.lower().find(phrase.lower())

        # Show a short amount of text around the exact phrase
        start = max(0, position - 300)
        end = min(len(text), position + len(phrase) + 500)

        snippet = text[start:end].replace("\n", " ")

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: mobile_phone_requirement
Matching chunks: 14

PROTO_001_P001_C001 | Page 1
1 Panda R, et al. BMJ Open 2022;12:e048628. doi:10.1136/bmjopen-2021-048628 Open access   Exploratory randomised trial of face- to-   face and mobile phone counselling  against usual care for tobacco cessation  in Indian primary care: a randomised  controlled trial protocol for  project   CERTAIN Rajmohan Panda,1 Rumana Omar,2 Rachael Hunter    ,3 Rajath R Prabhu,1  Arti Mishra,4 Irwin Nazareth    3 To cite: Panda R, Omar R,  Hunter R, et al.  Exploratory  randomised trial of face-   to-  face and mobile phone  counselling a gainst usual care  for tobacco cessation in Indian  primary care: a randomised  controlled trial protocol for  project CERTA

PROTO_001_P001_C002 | Page 1
) 2022. Re-  use  permitted under CC BY .  Published by BMJ. ABSTRACT Introduction Despite widespread use of smokeless  tobacco products by people within the Indian subcontinent,   there is little awareness among Indians of its hea

## 8. Inspect selected CERTAIN evidence

The broad searches identified several promising facts, but some search terms were too common or affected by PDF formatting.

We will now inspect a small set of specific chunks that contain the clearest evidence for the CERTAIN evaluation questions. These passages will be manually verified before the questions and gold labels are added to the evaluation dataset.

In [9]:
# Select the specific CERTAIN chunks we want to verify
selected_certain_chunk_ids = [
    "PROTO_001_P001_C002",
    "PROTO_001_P001_C004",
    "PROTO_001_P003_C002",
    "PROTO_001_P005_C001",
    "PROTO_001_P006_C002",
]


selected_certain_chunks = certain_chunks[
    certain_chunks["chunk_id"].isin(
        selected_certain_chunk_ids
    )
].copy()


# Print each selected chunk in full for manual verification
for _, row in selected_certain_chunks.iterrows():

    print()
    print("Chunk ID:", row["chunk_id"])
    print("Page:", row["page_number"])
    print()
    print(row["text"])
    print()


Chunk ID: PROTO_001_P001_C002
Page: 1

phfi.
 
org
Protocol
© Author(s) (or their 
employer(s)) 2022. Re-
 use 
permitted under CC BY
. 
Published by BMJ.
ABSTRACT
Introduction Despite widespread use of smokeless 
tobacco products by people within the Indian subcontinent,
 
there is little awareness among Indians of its health 
hazards when compared with smoked tobacco. We 
hypothesise that mobile phone counselling will be 
feasible and effective for smokeless tobacco cessation 
intervention in India. This paper presents the protocol of 
the development and conduct of an exploratory trial before 
progression to a full randomised controlled trial.
Methods and analysis
 An explora
tory randomised 
controlled trial will be conducted in urban primary health 
centres in the state of Odisha, India. A total of 250 
smokeless tobacco users will be recruited to the study 
(125 in each arm). Participants in the intervention arm 
will receive routine care together with a face-
 to-
 face 
counse

## 9. Extract the exact CERTAIN evidence

The selected chunks contain the information we need, but printing complete chunks produces unnecessary output.

We will now extract only the short passages that directly support each candidate evaluation question. These passages will be used to manually verify the gold-standard answers and evidence locations.

In [10]:
# Define the exact facts we want to verify
certain_gold_candidates = [
    {
        "topic": "follow_up_duration",
        "chunk_id": "PROTO_001_P001_C002",
        "search_phrase": "followed up for",
    },
    {
        "topic": "mobile_phone_requirement",
        "chunk_id": "PROTO_001_P001_C004",
        "search_phrase": "mobile phone",
    },
    {
        "topic": "intervention_counselling",
        "chunk_id": "PROTO_001_P003_C002",
        "search_phrase": "single 10",
    },
    {
        "topic": "routine_care",
        "chunk_id": "PROTO_001_P003_C002",
        "search_phrase": "routine care",
    },
    {
        "topic": "randomisation",
        "chunk_id": "PROTO_001_P005_C001",
        "search_phrase": "Randomisation will be done",
    },
    {
        "topic": "blinding",
        "chunk_id": "PROTO_001_P005_C001",
        "search_phrase": "Staff collecting outcome data will be blinded",
    },
]


evidence_records = []


for candidate in certain_gold_candidates:

    # Find the exact chunk
    row = certain_chunks[
        certain_chunks["chunk_id"] == candidate["chunk_id"]
    ].iloc[0]

    text = row["text"]
    phrase = candidate["search_phrase"]

    # Find the phrase inside the chunk
    position = text.lower().find(
        phrase.lower()
    )

    if position == -1:

        snippet = "SEARCH PHRASE NOT FOUND"

    else:

        # Keep a short passage around the evidence
        start = max(
            0,
            position - 180,
        )

        end = min(
            len(text),
            position + len(phrase) + 420,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )


    evidence_records.append(
        {
            "topic": candidate["topic"],
            "chunk_id": candidate["chunk_id"],
            "page_number": int(row["page_number"]),
            "evidence_snippet": snippet,
        }
    )


certain_gold_evidence_df = pd.DataFrame(
    evidence_records
)


# Print each candidate in a compact format
for _, row in certain_gold_evidence_df.iterrows():

    print("\nTOPIC:", row["topic"])
    print("Chunk:", row["chunk_id"])
    print("Page:", row["page_number"])
    print("Evidence:", row["evidence_snippet"])


TOPIC: follow_up_duration
Chunk: PROTO_001_P001_C002
Page: 1
Evidence: d by advice and reminder  mobile messages. The control arm will receive routine  care, delivered by a primary care physician based on  ‘Ask’ and ‘Advice’. All participants will be followed up for  3  months from the first counselling session.  The primary  outcome of this trial is to assess the feasibility to carry out  a full randomised controlled trial. Ethics and dissemination  Ethical a pprovals were  obtained from the Institutional Ethics Committee of Public  Health Foundation of India, Health Ministry’s Screening  Committee, Odisha State Ethics Board and also from  University College London Research Et

TOPIC: mobile_phone_requirement
Chunk: PROTO_001_P001_C004
Page: 1
Evidence: he physicians. There is a chance of biased results if  physicians may deliver repeated face-  to-  face counselling services.  ► The inclusion criteria include SLT users having a  mobile phone, this may exclude a section of the SLT  u

## 10. Add the supported CERTAIN evaluation questions

Six new supported questions were created from manually verified CERTAIN evidence.

These questions do not reuse the primary-outcome or sample-size questions used during Day 3 development. Each question is linked to the exact chunk and page containing the expected evidence before the RAG system is evaluated.

In [11]:
# Define the six manually verified CERTAIN evaluation questions
certain_supported_questions = [
    {
        "question_id": "EVAL_001",
        "question": (
            "How long were participants in the CERTAIN study "
            "followed after the first counselling session?"
        ),
        "gold_chunk_ids": "PROTO_001_P001_C002",
        "gold_pages": "1",
        "gold_answer_notes": (
            "All participants were followed for 3 months "
            "from the first counselling session."
        ),
        "verification_notes": (
            "Manually verified from CERTAIN page 1."
        ),
    },
    {
        "question_id": "EVAL_002",
        "question": (
            "What personal technology were smokeless tobacco users "
            "required to have for inclusion in CERTAIN?"
        ),
        "gold_chunk_ids": "PROTO_001_P001_C004",
        "gold_pages": "1",
        "gold_answer_notes": (
            "Eligible smokeless tobacco users were required "
            "to have a mobile phone."
        ),
        "verification_notes": (
            "Manually verified from CERTAIN page 1."
        ),
    },
    {
        "question_id": "EVAL_003",
        "question": (
            "What face-to-face counselling component was added "
            "to routine care in the CERTAIN intervention?"
        ),
        "gold_chunk_ids": "PROTO_001_P003_C002",
        "gold_pages": "3",
        "gold_answer_notes": (
            "The intervention included a single 10-minute "
            "face-to-face session delivered by a practice-based "
            "counsellor using the 5A tobacco-cessation approach."
        ),
        "verification_notes": (
            "Manually verified from CERTAIN page 3."
        ),
    },
    {
        "question_id": "EVAL_004",
        "question": (
            "How were eligible CERTAIN participants randomised "
            "between the intervention and control arms?"
        ),
        "gold_chunk_ids": "PROTO_001_P005_C001",
        "gold_pages": "5",
        "gold_answer_notes": (
            "Randomisation occurred at the individual participant "
            "level, was stratified by study practice site, used "
            "random permuted blocks of size 4 to 10, and allocated "
            "participants 1:1 to intervention or control."
        ),
        "verification_notes": (
            "Manually verified from CERTAIN page 5."
        ),
    },
    {
        "question_id": "EVAL_005",
        "question": (
            "Which CERTAIN study arms received the short "
            "Ask-and-Advice routine-care component?"
        ),
        "gold_chunk_ids": "PROTO_001_P003_C002",
        "gold_pages": "3",
        "gold_answer_notes": (
            "The 1-to-2-minute routine-care component based on "
            "Ask and Advice was delivered to both the intervention "
            "and control arms."
        ),
        "verification_notes": (
            "Manually verified from CERTAIN page 3."
        ),
    },
    {
        "question_id": "EVAL_006",
        "question": (
            "How did CERTAIN reduce the risk that outcome assessment "
            "would be influenced by knowledge of treatment allocation?"
        ),
        "gold_chunk_ids": "PROTO_001_P005_C001",
        "gold_pages": "5",
        "gold_answer_notes": (
            "Staff collecting outcome data were kept separate from "
            "staff delivering the intervention and were blinded to "
            "group assignment."
        ),
        "verification_notes": (
            "Manually verified from CERTAIN page 5."
        ),
    },
]


# Add the verified information to the evaluation blueprint
for record in certain_supported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display the six completed CERTAIN supported questions
print(
    evaluation_df.loc[
        evaluation_df["question_id"].isin(
            [f"EVAL_{number:03d}" for number in range(1, 7)]
        ),
        [
            "question_id",
            "question_type",
            "question",
            "gold_chunk_ids",
            "gold_pages",
        ],
    ].to_string(index=False)
)

question_id   question_type                                                                                                          question      gold_chunk_ids gold_pages
   EVAL_001 straightforward                     How long were participants in the CERTAIN study followed after the first counselling session? PROTO_001_P001_C002          1
   EVAL_002 straightforward                  What personal technology were smokeless tobacco users required to have for inclusion in CERTAIN? PROTO_001_P001_C004          1
   EVAL_003       difficult                    What face-to-face counselling component was added to routine care in the CERTAIN intervention? PROTO_001_P003_C002          3
   EVAL_004       difficult                      How were eligible CERTAIN participants randomised between the intervention and control arms? PROTO_001_P005_C001          5
   EVAL_005      confusable                                Which CERTAIN study arms received the short Ask-and-Advice routine-care comp

## 11. Verify unsupported CERTAIN questions

Unsupported evaluation questions should still be relevant to the protocol rather than obviously unrelated.

We will inspect the CERTAIN corpus for reported tobacco-cessation outcomes and final cost-effectiveness results. The protocol may discuss these topics as planned outcomes or future objectives, but the evaluation questions will only be labelled unsupported if no actual completed-trial result is documented.

In [12]:
# Search CERTAIN for evidence that might accidentally answer
# our proposed unsupported questions
unsupported_search_terms = [
    "tobacco cessation",
    "cost-effectiveness",
    "results",
    "quit",
    "completed",
]


for term in unsupported_search_terms:

    matches = certain_chunks[
        certain_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTERM: {term}")
    print("Matching chunks:", len(matches))

    # Show only short contexts around the first few matches
    for _, row in matches.head(3).iterrows():

        text = row["text"]

        position = text.lower().find(
            term.lower()
        )

        start = max(
            0,
            position - 180,
        )

        end = min(
            len(text),
            position + len(term) + 350,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TERM: tobacco cessation
Matching chunks: 14

PROTO_001_P001_C001 | Page 1
t al. BMJ Open 2022;12:e048628. doi:10.1136/bmjopen-2021-048628 Open access   Exploratory randomised trial of face- to-   face and mobile phone counselling  against usual care for tobacco cessation  in Indian primary care: a randomised  controlled trial protocol for  project   CERTAIN Rajmohan Panda,1 Rumana Omar,2 Rachael Hunter    ,3 Rajath R Prabhu,1  Arti Mishra,4 Irwin Nazareth    3 To cite: Panda R, Omar R,  Hunter R, et al.  Exploratory  randomised trial of face-   to-  face and mobile phone  counselling a gainst usual care  for tobac

PROTO_001_P001_C002 | Page 1
ittle awareness among Indians of its health  hazards when compared with smoked tobacco. We  hypothesise that mobile phone counselling will be  feasible and effective for smokeless tobacco cessation  intervention in India. This paper presents the protocol of  the development and conduct of an exploratory trial before  progression to a full random

## 12. Confirm that final CERTAIN outcomes are not reported

The protocol contains background results from previous studies and describes outcomes that the CERTAIN trial plans to evaluate.

Before labelling the two result questions as unsupported, we will perform a focused search for language that would indicate actual completed CERTAIN outcome or economic results.

In [13]:
# Search for wording that could indicate actual completed CERTAIN results
result_check_terms = [
    "participants quit",
    "cessation rate",
    "quit rate",
    "achieved cessation",
    "were abstinent",
    "cost-effective",
    "incremental cost",
    "cost per daly",
]


for term in result_check_terms:

    matches = certain_chunks[
        certain_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(
        f"{term}: {len(matches)} matching chunks"
    )

participants quit: 0 matching chunks
cessation rate: 0 matching chunks
quit rate: 4 matching chunks
achieved cessation: 0 matching chunks
were abstinent: 0 matching chunks
cost-effective: 0 matching chunks
incremental cost: 0 matching chunks
cost per daly: 1 matching chunks


## 13. Inspect remaining result-like evidence

A small number of chunks contain the phrases `quit rate` and `cost per DALY`.

Before labelling the final CERTAIN outcome questions as unsupported, we will inspect these passages to determine whether they report actual CERTAIN results or only describe planned analyses, outcomes, or background research.

In [14]:
# Inspect only the remaining result-like phrases
terms_to_inspect = [
    "quit rate",
    "cost per daly",
]


for term in terms_to_inspect:

    matches = certain_chunks[
        certain_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTERM: {term}")

    for _, row in matches.iterrows():

        text = row["text"]

        position = text.lower().find(
            term.lower()
        )

        start = max(
            0,
            position - 250,
        )

        end = min(
            len(text),
            position + len(term) + 450,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TERM: quit rate

PROTO_001_P002_C002 | Page 2
text messaging added to  other smoking cessation interventions was more effective  than other individual smoking cessation interventions (RR  1.59, 95% C I 1.09 to 2.33; I 2=0%, four studies).17 Another  meta- a nalysis of 13 trials reported smoking quit rates with  text messaging intervention were 35% higher than quit  rates for controls (OR=1.35, 95% C I 1.23 to 1.49).18 Other  reviews of studies from HICs suggest that such interven- tions can increase the chance of quitting smoking tobacco  from 39% to 80%. 19 20  Studies on SLT cessation that  included randomised controlled trials conducted in HICs  as well as some LMICs showed that behavioural cessation  interventions led to quit rates between 9

PROTO_001_P002_C003 | Page 2
as well as some LMICs showed that behavioural cessation  interventions led to quit rates between 9% and 51.5% at  6 m onths.11 21 There are, however, little data on the efficacy and effectiveness of such intervent

## 14. Add the unsupported CERTAIN evaluation questions

Two protocol-relevant questions were verified as unsupported.

The CERTAIN document discusses tobacco-cessation results from earlier studies and describes planned economic analyses, but it does not report completed CERTAIN quit-rate or cost-effectiveness results.

The expected response for both questions is therefore `Insufficient Evidence`.

In [15]:
# Define the two verified unsupported CERTAIN questions
certain_unsupported_questions = [
    {
        "question_id": "EVAL_007",
        "question": (
            "What proportion of CERTAIN participants actually quit "
            "smokeless tobacco by the end of the study?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol contains quit-rate results from previous studies "
            "but does not report a completed CERTAIN quit rate."
        ),
    },
    {
        "question_id": "EVAL_008",
        "question": (
            "What were the final cost-effectiveness results of CERTAIN?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol describes planned economic evaluation, including "
            "the feasibility of calculating cost per DALY and cost per QALY, "
            "but does not report final CERTAIN economic results."
        ),
    },
]


# Add the unsupported questions to the evaluation dataframe
for record in certain_unsupported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display all eight CERTAIN evaluation questions
certain_eval = evaluation_df[
    evaluation_df["document_name"] == "CERTAIN"
].copy()


print(
    certain_eval[
        [
            "question_id",
            "question_type",
            "expected_support",
            "question",
            "gold_chunk_ids",
            "gold_pages",
            "gold_answer_notes",
        ]
    ].to_string(index=False)
)

question_id   question_type expected_support                                                                                                          question      gold_chunk_ids gold_pages                                                                                                                                                                                          gold_answer_notes
   EVAL_001 straightforward        supported                     How long were participants in the CERTAIN study followed after the first counselling session? PROTO_001_P001_C002          1                                                                                                                            All participants were followed for 3 months from the first counselling session.
   EVAL_002 straightforward        supported                  What personal technology were smokeless tobacco users required to have for inclusion in CERTAIN? PROTO_001_P001_C004          1                         

## 15. Find candidate evidence in CARE_STROKE

We will now build the evaluation questions for the CARE_STROKE protocol.

The inclusion-criteria and primary-outcome questions used during Day 3 development will not be reused in the final held-out evaluation.

We will first identify other parts of the protocol that could support new questions before manually verifying the exact evidence.

In [16]:
# Keep only CARE_STROKE chunks
care_stroke_chunks = (
    chunks_df[
        chunks_df["document_name"] == "CARE_STROKE"
    ]
    .copy()
    .reset_index(drop=True)
)


# Search for different protocol topics
care_stroke_candidate_terms = [
    "random",
    "intervention",
    "control",
    "follow",
    "baseline",
    "secondary",
    "blind",
    "caregiver",
    "assessment",
    "sample size",
]


for term in care_stroke_candidate_terms:

    matches = care_stroke_chunks[
        care_stroke_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTERM: {term}")
    print("Matching chunks:", len(matches))

    if not matches.empty:

        print(
            matches[
                [
                    "chunk_id",
                    "page_number",
                ]
            ]
            .head(8)
            .to_string(index=False)
        )


TERM: random
Matching chunks: 9
           chunk_id  page_number
PROTO_002_P001_C001            1
PROTO_002_P001_C002            1
PROTO_002_P001_C003            1
PROTO_002_P002_C003            2
PROTO_002_P004_C002            4
PROTO_002_P005_C001            5
PROTO_002_P005_C002            5
PROTO_002_P006_C004            6

TERM: intervention
Matching chunks: 18
           chunk_id  page_number
PROTO_002_P001_C001            1
PROTO_002_P001_C002            1
PROTO_002_P001_C003            1
PROTO_002_P001_C004            1
PROTO_002_P002_C001            2
PROTO_002_P002_C002            2
PROTO_002_P002_C003            2
PROTO_002_P002_C004            2

TERM: control
Matching chunks: 8
           chunk_id  page_number
PROTO_002_P001_C001            1
PROTO_002_P001_C002            1
PROTO_002_P001_C003            1
PROTO_002_P002_C003            2
PROTO_002_P004_C001            4
PROTO_002_P004_C002            4
PROTO_002_P004_C003            4
PROTO_002_P005_C001            5

T

## 16. Inspect specific CARE_STROKE evidence

The broad search identified several useful areas of the CARE_STROKE protocol, but some terms occur in many chunks.

We will now inspect short passages around a smaller set of candidate topics. This will help us identify exact facts that can become new held-out evaluation questions.

In [17]:
# Define candidate CARE_STROKE topics to inspect
care_stroke_evidence_terms = [
    "randomisation",
    "caregiver",
    "follow-up",
    "secondary outcome",
    "control group",
    "blinded",
    "assessment",
    "usual care",
]


for term in care_stroke_evidence_terms:

    matches = care_stroke_chunks[
        care_stroke_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {term}")
    print("Matching chunks:", len(matches))

    # Show only the first two useful matches
    for _, row in matches.head(2).iterrows():

        text = row["text"]

        # Find the first occurrence of the term
        position = text.lower().find(
            term.lower()
        )

        start = max(
            0,
            position - 250,
        )

        end = min(
            len(text),
            position + len(term) + 450,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: randomisation
Matching chunks: 3

PROTO_002_P004_C002 | Page 4
e next of  kin if the participant is unable to consent. An entry form will be used to collect baseline information including the contact details of the participant  and the identified caregiver. This information will be  forwarded to the independent randomisation centre and  the participants eligible for inclusion will be randomised  to the intervention or control arm in a 1:1 ratio using a  secure, central, password-protected, web-based system.  The intervention will be started within 24  hours of  randomisation. sAMP l E  s I z E   E st IMAt I on The two main factors that determine the number of partic- ipants needed in this trial are the estimated event rate and  the size of the treatment effect. The 

PROTO_002_P005_C001 | Page 5
goals set by the specific therapist or a reha- bilitation team. o ut C o ME   MEA sur E s Primary outcome The primary outcome measure is dependency in activi- ties of daily living and w

## 17. Verify specific CARE_STROKE evidence

The broad search identified several useful areas of the CARE_STROKE protocol.

We will now inspect short passages around specific facts so that each supported evaluation question can be linked to exact source evidence before the RAG system is tested.

In [18]:
# Define specific CARE_STROKE facts to inspect
care_stroke_gold_candidates = [
    {
        "topic": "randomisation",
        "search_phrase": "randomised to the intervention or control arm",
    },
    {
        "topic": "intervention_start",
        "search_phrase": "within 24",
    },
    {
        "topic": "follow_up_assessment",
        "search_phrase": "6 weeks after randomisation",
    },
    {
        "topic": "secondary_outcomes",
        "search_phrase": "Secondary outcome measures will be",
    },
    {
        "topic": "adverse_events",
        "search_phrase": "Adverse events are very common",
    },
    {
        "topic": "caregiver",
        "search_phrase": "identified caregiver",
    },
]


for candidate in care_stroke_gold_candidates:

    topic = candidate["topic"]
    phrase = candidate["search_phrase"]

    matches = care_stroke_chunks[
        care_stroke_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    for _, row in matches.head(2).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 220,
        )

        end = min(
            len(text),
            position + len(phrase) + 500,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: randomisation
Matching chunks: 0

TOPIC: intervention_start
Matching chunks: 1

PROTO_002_P004_C002 | Page 4
centre and  the participants eligible for inclusion will be randomised  to the intervention or control arm in a 1:1 ratio using a  secure, central, password-protected, web-based system.  The intervention will be started within 24  hours of  randomisation. sAMP l E  s I z E   E st IMAt I on The two main factors that determine the number of partic- ipants needed in this trial are the estimated event rate and  the size of the treatment effect. The primary outcome for  the ‘Care for Stroke’ trial is dependency in activities of  daily living measured at 6  weeks postrecruitment. Estimated  event rate: in a meta-analysis of early  supported discharge trial among participants with stroke,  50% of the stroke survivors were eith

TOPIC: follow_up_assessment
Matching chunks: 0

TOPIC: secondary_outcomes
Matching chunks: 0

TOPIC: adverse_events
Matching chunks: 0

TOPIC: caregiver

## 17. Verify specific CARE_STROKE evidence

Some exact phrase searches failed because PDF extraction introduced irregular spacing and formatting.

We will therefore use shorter, distinctive search terms to locate candidate evidence. This search is only for evaluation design and does not modify the underlying protocol text.

The exact source passages will still be manually inspected before any question is added to the gold-standard evaluation set.

In [19]:
# Use short, distinctive phrases that are less sensitive
# to irregular PDF spacing
care_stroke_gold_candidates = {
    "randomisation": "1:1 ratio",
    "intervention_start": "within 24",
    "follow_up_timing": "6 weeks",
    "secondary_outcomes": "Secondary",
    "adverse_events": "Adverse",
    "consent_if_unable": "next of kin",
}


for topic, phrase in care_stroke_gold_candidates.items():

    # Find chunks containing the candidate phrase
    matches = care_stroke_chunks[
        care_stroke_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    # Show at most two short passages
    for _, row in matches.head(2).iterrows():

        text = row["text"]

        # Find where the phrase occurs
        position = text.lower().find(
            phrase.lower()
        )

        # Keep a manageable amount of surrounding evidence
        start = max(
            0,
            position - 250,
        )

        end = min(
            len(text),
            position + len(phrase) + 500,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: randomisation
Matching chunks: 1

PROTO_002_P004_C002 | Page 4
he contact details of the participant  and the identified caregiver. This information will be  forwarded to the independent randomisation centre and  the participants eligible for inclusion will be randomised  to the intervention or control arm in a 1:1 ratio using a  secure, central, password-protected, web-based system.  The intervention will be started within 24  hours of  randomisation. sAMP l E  s I z E   E st IMAt I on The two main factors that determine the number of partic- ipants needed in this trial are the estimated event rate and  the size of the treatment effect. The primary outcome for  the ‘Care for Stroke’ trial is dependency in activities of  daily living measured at 6  weeks postrecruitment. Estimated  event rate: in a meta-analysi

TOPIC: intervention_start
Matching chunks: 1

PROTO_002_P004_C002 | Page 4
the independent randomisation centre and  the participants eligible for inclusion will be ran

## 18. Inspect remaining CARE_STROKE candidate evidence

Several strong CARE_STROKE facts have already been identified, including the randomisation method, intervention start time, and six-week assessment timing.

We will now inspect the remaining candidate evidence for secondary outcomes, adverse events, and intervention delivery so that the final six supported questions cover different parts of the protocol.

In [20]:
# Inspect only the remaining CARE_STROKE topics we need
remaining_care_stroke_terms = {
    "secondary_outcomes": "Modified Barthel Index",
    "adverse_events": "Death due to any vascular causes",
    "intervention_delivery": "smartphone",
}


for topic, phrase in remaining_care_stroke_terms.items():

    matches = care_stroke_chunks[
        care_stroke_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    for _, row in matches.head(3).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 300,
        )

        end = min(
            len(text),
            position + len(phrase) + 650,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: secondary_outcomes
Matching chunks: 1

PROTO_002_P005_C001 | Page 5
uffered  a stroke in six categories. The scores range from 0 (no  symptoms) to a maximum of 6   (dead). A dichotomous  approach to outcome analysis will be used. Participants’  scores will be categorised into MRS scores of 0–3 and  4–6. sEC ondA ry out C o ME Secondar y outcome measures will be:  ► Modified Barthel Index.24  ► Modified Caregiver Strain Index.28  ► Quality of Life measured by  World Health Organiza- tion - Quality of Life Brief    (WHOQOL–BREF).29  ► Use of healthcare and rehabilitation services (therapy,  hospitalisation and medication, AYUSH, traditional  practices and so on). This information will be collected through question- naire at baseline and after 6 weeks. The smartphone

TOPIC: adverse_events
Matching chunks: 1

PROTO_002_P005_C002 | Page 5
e that could occur due to subse- quent stroke events that are unrelated to the trial. 24 This  will also allow accurate assessment of the outcome

## 19. Add the supported CARE_STROKE evaluation questions

Six new supported questions were created from manually verified CARE_STROKE evidence.

The Day 3 questions about inclusion criteria and the primary outcome are not reused. Where overlapping chunks independently contain sufficient evidence, multiple acceptable gold chunks are recorded so that retrieval is not unfairly penalised for returning an equivalent source chunk.

In [21]:
# Define the six manually verified CARE_STROKE questions
care_stroke_supported_questions = [
    {
        "question_id": "EVAL_009",
        "question": (
            "How were eligible CARE_STROKE participants allocated "
            "between the intervention and control groups?"
        ),
        "gold_chunk_ids": "PROTO_002_P004_C002",
        "gold_pages": "4",
        "gold_answer_notes": (
            "Participants were randomised 1:1 to intervention or control "
            "using a secure, central, password-protected web-based system."
        ),
        "verification_notes": (
            "Manually verified from CARE_STROKE page 4."
        ),
    },
    {
        "question_id": "EVAL_010",
        "question": (
            "How soon after randomisation was the CARE_STROKE "
            "intervention started?"
        ),
        "gold_chunk_ids": "PROTO_002_P004_C002",
        "gold_pages": "4",
        "gold_answer_notes": (
            "The intervention was started within 24 hours of randomisation."
        ),
        "verification_notes": (
            "Manually verified from CARE_STROKE page 4."
        ),
    },
    {
        "question_id": "EVAL_011",
        "question": (
            "At what time points was questionnaire information collected "
            "during CARE_STROKE follow-up?"
        ),
        "gold_chunk_ids": (
            "PROTO_002_P005_C001;"
            "PROTO_002_P005_C002"
        ),
        "gold_pages": "5",
        "gold_answer_notes": (
            "Questionnaire information was collected at baseline "
            "and again after 6 weeks."
        ),
        "verification_notes": (
            "Manually verified from overlapping CARE_STROKE page 5 chunks."
        ),
    },
    {
        "question_id": "EVAL_012",
        "question": (
            "Which secondary outcome measures were specified "
            "in the CARE_STROKE protocol?"
        ),
        "gold_chunk_ids": "PROTO_002_P005_C001",
        "gold_pages": "5",
        "gold_answer_notes": (
            "Secondary outcomes included the Modified Barthel Index, "
            "Modified Caregiver Strain Index, WHOQOL-BREF, and use of "
            "healthcare and rehabilitation services."
        ),
        "verification_notes": (
            "Manually verified from CARE_STROKE page 5."
        ),
    },
    {
        "question_id": "EVAL_013",
        "question": (
            "Was CARE_STROKE simply a smartphone application, "
            "or how was the intervention described?"
        ),
        "gold_chunk_ids": (
            "PROTO_002_P001_C001;"
            "PROTO_002_P002_C002"
        ),
        "gold_pages": "1;2",
        "gold_answer_notes": (
            "CARE_STROKE was described as a smartphone-enabled, "
            "carer-supported educational intervention for managing "
            "physical disabilities following stroke."
        ),
        "verification_notes": (
            "Manually verified from CARE_STROKE pages 1 and 2."
        ),
    },
    {
        "question_id": "EVAL_014",
        "question": (
            "What adverse events were specifically expected "
            "during the CARE_STROKE trial?"
        ),
        "gold_chunk_ids": "PROTO_002_P005_C002",
        "gold_pages": "5",
        "gold_answer_notes": (
            "Expected adverse events included death from vascular causes, "
            "hospitalisation for post-stroke complications, and occurrence "
            "of a secondary stroke."
        ),
        "verification_notes": (
            "Manually verified from CARE_STROKE page 5."
        ),
    },
]


# Add the verified CARE_STROKE questions to the evaluation dataframe
for record in care_stroke_supported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display the six completed supported questions
print(
    evaluation_df.loc[
        evaluation_df["question_id"].isin(
            [f"EVAL_{number:03d}" for number in range(9, 15)]
        ),
        [
            "question_id",
            "question_type",
            "question",
            "gold_chunk_ids",
            "gold_pages",
        ],
    ].to_string(index=False)
)

question_id   question_type                                                                                          question                          gold_chunk_ids gold_pages
   EVAL_009 straightforward How were eligible CARE_STROKE participants allocated between the intervention and control groups?                     PROTO_002_P004_C002          4
   EVAL_010 straightforward                            How soon after randomisation was the CARE_STROKE intervention started?                     PROTO_002_P004_C002          4
   EVAL_011       difficult         At what time points was questionnaire information collected during CARE_STROKE follow-up? PROTO_002_P005_C001;PROTO_002_P005_C002          5
   EVAL_012       difficult                      Which secondary outcome measures were specified in the CARE_STROKE protocol?                     PROTO_002_P005_C001          5
   EVAL_013      confusable           Was CARE_STROKE simply a smartphone application, or how was the intervention 

## 20. Verify unsupported CARE_STROKE questions

The unsupported questions should remain closely related to the trial.

We will check whether the protocol reports actual completed functional or economic results. Planned outcome measurements, sample-size assumptions, pilot-study findings, and results from previous research do not count as final CARE_STROKE trial results.

The questions will only be labelled unsupported after confirming that the protocol does not contain the requested completed-trial findings.

In [22]:
# Search for wording that could indicate completed CARE_STROKE results
care_stroke_result_terms = [
    "were independent",
    "achieved independence",
    "improved functional",
    "at 6 weeks",
    "cost-effective",
    "incremental cost",
    "cost per",
    "results showed",
]


for term in care_stroke_result_terms:

    matches = care_stroke_chunks[
        care_stroke_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(
        f"{term}: {len(matches)} matching chunks"
    )

were independent: 0 matching chunks
achieved independence: 0 matching chunks
improved functional: 0 matching chunks
at 6 weeks: 0 matching chunks
cost-effective: 3 matching chunks
incremental cost: 0 matching chunks
cost per: 0 matching chunks
results showed: 0 matching chunks


## 21. Inspect CARE_STROKE cost-effectiveness evidence

The functional-result searches did not identify completed CARE_STROKE outcome findings.

The term `cost-effective` appears in several chunks, so we will inspect those passages before deciding whether the final cost-effectiveness question can safely be labelled unsupported.

In [23]:
# Inspect every CARE_STROKE chunk containing "cost-effective"
cost_effective_matches = care_stroke_chunks[
    care_stroke_chunks["text"].str.contains(
        "cost-effective",
        case=False,
        na=False,
        regex=False,
    )
]


for _, row in cost_effective_matches.iterrows():

    text = row["text"]

    # Find where the phrase appears
    position = text.lower().find(
        "cost-effective"
    )

    # Show a short passage around the match
    start = max(
        0,
        position - 300,
    )

    end = min(
        len(text),
        position + 700,
    )

    snippet = (
        text[start:end]
        .replace("\n", " ")
    )

    print()
    print(
        f"{row['chunk_id']} | Page {row['page_number']}"
    )
    print(snippet)


PROTO_002_P002_C001 | Page 2
gies have developed  various solutions to meet the needs of stroke survivors,  the best way to use this approach in stroke rehabilita- tion is also still unclear. 10 There is insufficient evidence  for tele-rehabilitation services. 11 This context provides a  strong grounding for the development of cost-effective  multidimensional stroke rehabilitation interventions to  meet the demands of the stroke survivors. In the absence  of organised stroke care services and with the limited  resources available for rehabilitation, a comprehensive  approach to address the growing burden of stroke-related  disability in India becomes pertinent. 12 This approach  could be pivotal in integrating various strategies for reha- bilitation3 (educational, community-based rehabilitation,  digital technology, self/supported management and so  on). It could also be useful for targeting the full range  of impacts of stroke, including on impairments, activity  limitations and part

## 22. Add the unsupported CARE_STROKE evaluation questions

Two CARE_STROKE questions were verified as unsupported.

The protocol describes planned functional and economic evaluations, but it does not report completed CARE_STROKE outcome or cost-effectiveness results.

The expected response for these questions is therefore `Insufficient Evidence`.

In [24]:
care_stroke_unsupported_questions = [
    {
        "question_id": "EVAL_015",
        "question": (
            "What proportion of CARE_STROKE participants achieved "
            "improved functional independence at the six-week assessment?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol specifies functional outcome measurement at "
            "6 weeks but does not report completed CARE_STROKE results."
        ),
    },
    {
        "question_id": "EVAL_016",
        "question": (
            "What were the final cost-effectiveness results of CARE_STROKE?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol describes planned cost collection and "
            "cost-effectiveness evaluation but does not report final "
            "CARE_STROKE economic results."
        ),
    },
]


# Add the unsupported questions to the evaluation dataframe
for record in care_stroke_unsupported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display all eight CARE_STROKE evaluation questions
care_stroke_eval = evaluation_df[
    evaluation_df["document_name"] == "CARE_STROKE"
].copy()


print(
    care_stroke_eval[
        [
            "question_id",
            "question_type",
            "expected_support",
            "question",
            "gold_chunk_ids",
            "gold_pages",
            "gold_answer_notes",
        ]
    ].to_string(index=False)
)

question_id   question_type expected_support                                                                                                          question                          gold_chunk_ids gold_pages                                                                                                                                        gold_answer_notes
   EVAL_009 straightforward        supported                 How were eligible CARE_STROKE participants allocated between the intervention and control groups?                     PROTO_002_P004_C002          4                                Participants were randomised 1:1 to intervention or control using a secure, central, password-protected web-based system.
   EVAL_010 straightforward        supported                                            How soon after randomisation was the CARE_STROKE intervention started?                     PROTO_002_P004_C002          4                                                                 

## 23. Identify INTEGRA evaluation evidence

We will now build the held-out evaluation questions for the INTEGRA protocol.

The exclusion-criteria and follow-up questions used during Day 3 development will not be reused.

To keep the evaluation process efficient, we will inspect a focused set of potentially useful protocol topics and identify exact evidence before creating the gold-standard questions.

In [25]:
# Keep only INTEGRA chunks
integra_chunks = (
    chunks_df[
        chunks_df["document_name"] == "INTEGRA"
    ]
    .copy()
    .reset_index(drop=True)
)


# Use focused terms representing different parts of the protocol
integra_candidate_terms = {
    "randomisation": "random",
    "intervention_groups": "Intervention group",
    "coaching": "coaching",
    "sms_messages": "SMS",
    "monographic_consultation": "monographic",
    "questionnaires": "questionnaire",
    "hba1c": "HbA1c",
    "sample_size": "sample size",
}


for topic, phrase in integra_candidate_terms.items():

    matches = integra_chunks[
        integra_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    # Show only the first two short evidence passages
    for _, row in matches.head(2).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 250,
        )

        end = min(
            len(text),
            position + len(phrase) + 500,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: randomisation
Matching chunks: 8

PROTO_003_P001_C001 | Page 1
 Intervention Group 1 and 2. Each intervention group will recruit 216 participants (the same as in the control group) between the ages of 30 and 80 years with deficient glycaemic control (HbA1c > 9%). The control group will be established based on a randomized selection from the large SIDIAP

PROTO_003_P001_C002 | Page 1
(the same as in the control group) between the ages of 30 and 80 years with deficient glycaemic control (HbA1c > 9%). The control group will be established based on a randomized selection from the large SIDIAP (Sistema d ’Informació per al desenvolupament de la Investigació en Atenció Primària) database of patients with comparable socio-demographic and clinical characteristics from the three provinces. Discussion: This study is a comprehensive, pragmatic intervention based on glycaemic treatment intensification and the control of other cardiovascular risk factors. It is also aimed at improving treat

## 24. Verify specific INTEGRA evidence

The focused search identified several strong candidate facts from different parts of the INTEGRA protocol.

We will now inspect short passages for six specific facts before adding them to the gold-standard evaluation set. The Day 3 exclusion-criteria and follow-up questions will remain excluded from the held-out metrics.

In [26]:
# Define six specific INTEGRA facts to verify
integra_gold_candidates = {
    "hba1c_threshold": "HbA1c > 9%",
    "control_group_selection": "randomized selection",
    "coaching_training": "7-h training",
    "sms_intervention": "SMS",
    "monographic_consultation": "monographic",
    "final_sample_size": "648 study subjects",
}


for topic, phrase in integra_gold_candidates.items():

    matches = integra_chunks[
        integra_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    for _, row in matches.head(3).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 280,
        )

        end = min(
            len(text),
            position + len(phrase) + 600,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: hba1c_threshold
Matching chunks: 6

PROTO_003_P001_C001 | Page 1
alonia (Spain), including 3 specific health care areas. The intervention study has two arms: Intervention Group 1 and 2. Each intervention group will recruit 216 participants (the same as in the control group) between the ages of 30 and 80 years with deficient glycaemic control (HbA1c > 9%). The control group will be established based on a randomized selection from the large SIDIAP

PROTO_003_P001_C002 | Page 1
(the same as in the control group) between the ages of 30 and 80 years with deficient glycaemic control (HbA1c > 9%). The control group will be established based on a randomized selection from the large SIDIAP (Sistema d ’Informació per al desenvolupament de la Investigació en Atenció Primària) database of patients with comparable socio-demographic and clinical characteristics from the three provinces. Discussion: This study is a comprehensive, pragmatic intervention based on glycaemic treatment intensifica

## 25. Verify remaining INTEGRA intervention evidence

Three strong INTEGRA facts have already been verified: the HbA1c threshold, control-group selection, and final calculated sample size.

We will now verify the coaching, monographic-consultation, and SMS intervention details before adding the six supported INTEGRA questions to the gold-standard evaluation set.

In [27]:
# Verify the remaining INTEGRA intervention facts
remaining_integra_facts = {
    "coaching_training": "7-h training",
    "monographic_consultation": "without the monographic consultation",
    "sms_messages": "SMS phone messages",
}


for topic, phrase in remaining_integra_facts.items():

    matches = integra_chunks[
        integra_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    for _, row in matches.head(3).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 300,
        )

        end = min(
            len(text),
            position + len(phrase) + 650,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: coaching_training
Matching chunks: 1

PROTO_003_P005_C001 | Page 5
o clinical practice guidelines, thus broadening the professionals ’ knowledge base with regard to diabetes by providing them with an increased degree of autonomy in diabetes case management. – Training of professionals in coaching: all professionals in the primary care centres will participate in a 7-h training programme for coaching to be able to impart practical theoretical content on the following subjects: strategies for active listening; strategies for communication without value judgement; support strategies to develop self- management skills for diabetes, hypertension, and hyperlipidaemia; strategies to provide social and emotional support; strategies to motivate lifestyle changes; strategies for medication adherence; and strategies to access community resources. – The professionals will attend a 2-h training programme to update their training for reviewing the practical cases discussed in consultation us

## 26. Verify the INTEGRA monographic-consultation difference

The coaching and SMS intervention details were verified successfully.

The exact search phrase for the monographic consultation did not match because of PDF formatting, so we will use the shorter term `monographic` and inspect the surrounding evidence before adding the final supported INTEGRA questions.

In [28]:
# Find INTEGRA chunks mentioning the monographic consultation
monographic_matches = integra_chunks[
    integra_chunks["text"].str.contains(
        "monographic",
        case=False,
        na=False,
        regex=False,
    )
]


print("Matching chunks:", len(monographic_matches))


for _, row in monographic_matches.iterrows():

    text = row["text"]

    # Find the first occurrence of the word
    position = text.lower().find(
        "monographic"
    )

    # Show a short passage around the evidence
    start = max(
        0,
        position - 300,
    )

    end = min(
        len(text),
        position + 750,
    )

    snippet = (
        text[start:end]
        .replace("\n", " ")
    )

    print()
    print(
        f"{row['chunk_id']} | Page {row['page_number']}"
    )
    print(snippet)

Matching chunks: 8

PROTO_003_P002_C002 | Page 2
s goal is to use the mixed methodology by previously ex- ploring the patients ’ own perspective, in order to design a proper implementation strategy. Previous studies have shown that specialised Diabetes Unit improve glycaemic control [ 11]. Our main intervention was designed to evaluate if a local monographic consultation run by pri- mary healthcare professionals could be effective in the context of real-world primary healthcare practice for the management of very poor controlled diabetic patients.

PROTO_003_P002_C003 | Page 2
a proper implementation strategy. Previous studies have shown that specialised Diabetes Unit improve glycaemic control [ 11]. Our main intervention was designed to evaluate if a local monographic consultation run by pri- mary healthcare professionals could be effective in the context of real-world primary healthcare practice for the management of very poor controlled diabetic patients. Methods/design Aims of the 

## 27. Add the supported INTEGRA evaluation questions

Six new supported questions were created from manually verified INTEGRA evidence.

The exclusion-criteria and follow-up questions used during Day 3 development are not reused. Where multiple overlapping chunks independently support the same fact, all acceptable gold chunks are recorded.

In [29]:
# Define the six manually verified INTEGRA questions
integra_supported_questions = [
    {
        "question_id": "EVAL_017",
        "question": (
            "What HbA1c threshold was used to define poor glycaemic "
            "control for participants in INTEGRA?"
        ),
        "gold_chunk_ids": (
            "PROTO_003_P001_C001;"
            "PROTO_003_P001_C002;"
            "PROTO_003_P003_C002"
        ),
        "gold_pages": "1;3",
        "gold_answer_notes": (
            "Poor glycaemic control was defined as HbA1c greater than 9%."
        ),
        "verification_notes": (
            "Manually verified from INTEGRA pages 1 and 3."
        ),
    },
    {
        "question_id": "EVAL_018",
        "question": (
            "How long was the coaching training programme provided "
            "to primary-care professionals in INTEGRA?"
        ),
        "gold_chunk_ids": "PROTO_003_P005_C001",
        "gold_pages": "5",
        "gold_answer_notes": (
            "Primary-care professionals participated in a 7-hour "
            "coaching training programme."
        ),
        "verification_notes": (
            "Manually verified from INTEGRA page 5."
        ),
    },
    {
        "question_id": "EVAL_019",
        "question": (
            "How was the INTEGRA control group selected and matched "
            "to the intervention groups?"
        ),
        "gold_chunk_ids": (
            "PROTO_003_P001_C002;"
            "PROTO_003_P005_C002"
        ),
        "gold_pages": "1;5",
        "gold_answer_notes": (
            "The control group was selected randomly from the SIDIAP "
            "database using patients meeting the study criteria with "
            "comparable sociodemographic and clinical characteristics "
            "to the intervention groups."
        ),
        "verification_notes": (
            "Manually verified from INTEGRA pages 1 and 5."
        ),
    },
    {
        "question_id": "EVAL_020",
        "question": (
            "What total study sample was calculated to obtain "
            "statistical significance in INTEGRA?"
        ),
        "gold_chunk_ids": "PROTO_003_P008_C002",
        "gold_pages": "8",
        "gold_answer_notes": (
            "The final calculated sample was 648 study subjects."
        ),
        "verification_notes": (
            "Manually verified from INTEGRA page 8."
        ),
    },
    {
        "question_id": "EVAL_021",
        "question": (
            "What key intervention component distinguished IG-1 "
            "from IG-2 in INTEGRA?"
        ),
        "gold_chunk_ids": (
            "PROTO_003_P005_C001;"
            "PROTO_003_P005_C002;"
            "PROTO_003_P006_C003"
        ),
        "gold_pages": "5;6",
        "gold_answer_notes": (
            "IG-1 included the monographic consultation, while IG-2 "
            "received the otherwise similar personalised intervention "
            "without the monographic consultation."
        ),
        "verification_notes": (
            "Manually verified from INTEGRA pages 5 and 6."
        ),
    },
    {
        "question_id": "EVAL_022",
        "question": (
            "According to the INTEGRA study flowchart, which intervention "
            "groups included patient SMS phone messages?"
        ),
        "gold_chunk_ids": "PROTO_003_P006_C003",
        "gold_pages": "6",
        "gold_answer_notes": (
            "Both Intervention Group 1 and Intervention Group 2 included "
            "an intervention based on patient SMS phone messages."
        ),
        "verification_notes": (
            "Manually verified from the INTEGRA flowchart on page 6."
        ),
    },
]


# Add the verified INTEGRA questions to the evaluation dataframe
for record in integra_supported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display the six completed supported questions
print(
    evaluation_df.loc[
        evaluation_df["question_id"].isin(
            [f"EVAL_{number:03d}" for number in range(17, 23)]
        ),
        [
            "question_id",
            "question_type",
            "question",
            "gold_chunk_ids",
            "gold_pages",
        ],
    ].to_string(index=False)
)

question_id   question_type                                                                                                 question                                              gold_chunk_ids gold_pages
   EVAL_017 straightforward              What HbA1c threshold was used to define poor glycaemic control for participants in INTEGRA? PROTO_003_P001_C001;PROTO_003_P001_C002;PROTO_003_P003_C002        1;3
   EVAL_018 straightforward          How long was the coaching training programme provided to primary-care professionals in INTEGRA?                                         PROTO_003_P005_C001          5
   EVAL_019       difficult                       How was the INTEGRA control group selected and matched to the intervention groups?                     PROTO_003_P001_C002;PROTO_003_P005_C002        1;5
   EVAL_020       difficult                    What total study sample was calculated to obtain statistical significance in INTEGRA?                                         PROTO_003_P

## 28. Verify unsupported INTEGRA questions

The unsupported INTEGRA questions will focus on completed glycaemic-control results.

The protocol describes planned HbA1c comparisons and target levels, but these questions will only be labelled unsupported if the document does not report actual completed study outcomes.

In [30]:
# Search for wording that could indicate completed INTEGRA results
integra_result_terms = [
    "results showed",
    "significantly reduced",
    "mean reduction",
    "achieved hba1c",
    "patients achieved",
    "final hba1c",
    "at study completion",
    "was significantly",
]


for term in integra_result_terms:

    matches = integra_chunks[
        integra_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(
        f"{term}: {len(matches)} matching chunks"
    )

results showed: 0 matching chunks
significantly reduced: 0 matching chunks
mean reduction: 0 matching chunks
achieved hba1c: 0 matching chunks
patients achieved: 0 matching chunks
final hba1c: 0 matching chunks
at study completion: 0 matching chunks
was significantly: 0 matching chunks


## 29. Add the unsupported INTEGRA evaluation questions

Two INTEGRA questions were verified as unsupported.

The protocol describes planned comparisons of HbA1c and target glycaemic-control levels, but it does not report completed INTEGRA study results showing the final HbA1c reduction or the proportion of participants who achieved the target level.

The expected response for both questions is therefore `Insufficient Evidence`.

In [31]:
# Define the two verified unsupported INTEGRA questions
integra_unsupported_questions = [
    {
        "question_id": "EVAL_023",
        "question": (
            "What was the final reduction in HbA1c achieved by IG-1 "
            "compared with the control group in INTEGRA?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol describes planned HbA1c comparisons but does "
            "not report a completed IG-1 versus control HbA1c result."
        ),
    },
    {
        "question_id": "EVAL_024",
        "question": (
            "What proportion of INTEGRA participants ultimately "
            "achieved the target HbA1c level?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol discusses target HbA1c levels but does not "
            "report the final proportion of participants achieving them."
        ),
    },
]


# Add the unsupported questions to the evaluation dataframe
for record in integra_unsupported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display all eight INTEGRA evaluation questions
integra_eval = evaluation_df[
    evaluation_df["document_name"] == "INTEGRA"
].copy()


print(
    integra_eval[
        [
            "question_id",
            "question_type",
            "expected_support",
            "question",
            "gold_chunk_ids",
            "gold_pages",
            "gold_answer_notes",
        ]
    ].to_string(index=False)
)

question_id   question_type expected_support                                                                                                 question                                              gold_chunk_ids gold_pages                                                                                                                                                                                    gold_answer_notes
   EVAL_017 straightforward        supported              What HbA1c threshold was used to define poor glycaemic control for participants in INTEGRA? PROTO_003_P001_C001;PROTO_003_P001_C002;PROTO_003_P003_C002        1;3                                                                                                                                         Poor glycaemic control was defined as HbA1c greater than 9%.
   EVAL_018 straightforward        supported          How long was the coaching training programme provided to primary-care professionals in INTEGRA?               

## 30. Identify LISTEN evaluation evidence

We will now create the held-out evaluation questions for the LISTEN protocol.

The primary-outcome and sample-size questions used during Day 3 development will not be reused.

We will inspect several different areas of the protocol and identify exact evidence for six new supported questions before testing the RAG system.

In [32]:
# Keep only LISTEN chunks
listen_chunks = (
    chunks_df[
        chunks_df["document_name"] == "LISTEN"
    ]
    .copy()
    .reset_index(drop=True)
)


# Search different areas of the LISTEN protocol
listen_candidate_terms = {
    "randomisation": "random",
    "intervention": "intervention",
    "control": "control",
    "follow_up": "follow",
    "secondary_outcomes": "secondary outcome",
    "blinding": "blind",
    "eligibility": "eligible",
    "questionnaire": "questionnaire",
}


for topic, phrase in listen_candidate_terms.items():

    # Find matching LISTEN chunks
    matches = listen_chunks[
        listen_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    # Show only two short evidence passages
    for _, row in matches.head(2).iterrows():

        text = row["text"]

        # Find the matching term inside the chunk
        position = text.lower().find(
            phrase.lower()
        )

        # Keep only a short amount of surrounding text
        start = max(
            0,
            position - 250,
        )

        end = min(
            len(text),
            position + len(phrase) + 500,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: randomisation
Matching chunks: 20

PROTO_004_P001_C001 | Page 1
 data made available in this article, unless otherwise stated in a credit line to the data. Open Access Trials Effectiveness and cost-effectiveness  of a personalised self-management intervention  for living with long COVID: protocol  for the LISTEN randomised controlled trial Claire Potter1†, Fiona Leggat2,3†, Rachel Lowe1, Philip Pallmann1, Muhammad Riaz1, Christy Barlow1,  Adrian Edwards4,5, Aloysius Niroshan Siriwardena6, Nick Sevdalis7, Bernadette Sewell8, Jackie McRae2,3,  Jessica Fish9, Maria Ines de Sousa de Abreu10, Fiona Jones2,3,11 and Monica Busse1*    Abstract  Background Individuals living with long COVID experience multiple, interacting and fluctuating symptoms which  can have a dramatic impact on daily living. The aim of the Long 

PROTO_004_P001_C002 | Page 1
al participation, emotional  well-being, quality of life, fatigue, and self-efficacy. Cost-effectiveness will also be evaluated, and a detail

## 31. Verify specific LISTEN evidence

The broad LISTEN search identified several useful areas, but the output was too large to inspect reliably.

We will now search only for six specific facts that could form the supported held-out evaluation questions. This keeps the evidence review compact and makes it easier to record the correct gold chunks.

In [33]:
# Search only for the specific LISTEN facts we may use
listen_gold_candidates = {
    "questionnaire_timing": "6 weeks and 3 months",
    "self_referral": "self-refer",
    "language_support": "translator",
    "process_evaluation": "implementation scales",
    "usual_care": "usual care",
    "process_questionnaires": "Process evaluation questionnaires",
}


for topic, phrase in listen_gold_candidates.items():

    matches = listen_chunks[
        listen_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    for _, row in matches.head(3).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 300,
        )

        end = min(
            len(text),
            position + len(phrase) + 650,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: questionnaire_timing
Matching chunks: 1

PROTO_004_P003_C001 | Page 3
Potter et al. Trials           (2023) 24:75    Those who are eligible and consent to participating in  the study will complete a series of questionnaires at base- line and then repeat a selection at 6 weeks and 3 months  post randomisation (see supplementary materials). Participants and practitioners will be invited to consent  to taking part in process evaluation which will involve  implementation scales, interviews, and focus groups. Eligibility criteria Eligible participants are people who are aged 18 years  or older, are an English or Welsh speaker or have access  to someone who can act as a translator. They must have  experienced at least one long COVID symptom [4] for 12  weeks or longer and additionally meet at least one of the  following criteria: (i) positive SARS-CoV-2 PCR or antigen test (positive COVID-19 test) during the acute phase  of illness

TOPIC: self_referral
Matching chunks: 4

PROTO_004_

## 32. Add the supported LISTEN evaluation questions

Six supported LISTEN questions were created from manually verified evidence.

The primary-outcome and sample-size questions used during Day 3 development are excluded. The questions cover follow-up timing, recruitment, intervention delivery, process evaluation, eligibility, and structured table evidence.

In [34]:
# Define the six manually verified LISTEN questions
listen_supported_questions = [
    {
        "question_id": "EVAL_025",
        "question": (
            "At what time points were LISTEN participants asked "
            "to complete study questionnaires?"
        ),
        "gold_chunk_ids": (
            "PROTO_004_P001_C002;"
            "PROTO_004_P003_C001"
        ),
        "gold_pages": "1;3",
        "gold_answer_notes": (
            "Questionnaires were completed at baseline and repeated "
            "at 6 weeks and 3 months after randomisation."
        ),
        "verification_notes": (
            "Manually verified from LISTEN pages 1 and 3."
        ),
    },
    {
        "question_id": "EVAL_026",
        "question": (
            "How could eligible participants be referred or "
            "self-refer into the LISTEN trial?"
        ),
        "gold_chunk_ids": (
            "PROTO_004_P001_C002;"
            "PROTO_004_P002_C003;"
            "PROTO_004_P003_C002"
        ),
        "gold_pages": "1;2;3",
        "gold_answer_notes": (
            "Eligible participants could self-refer through the LISTEN "
            "website or be referred through long COVID services."
        ),
        "verification_notes": (
            "Manually verified from LISTEN recruitment evidence."
        ),
    },
    {
        "question_id": "EVAL_027",
        "question": (
            "What intervention support was offered to participants "
            "randomised to the LISTEN intervention?"
        ),
        "gold_chunk_ids": "PROTO_004_P001_C002",
        "gold_pages": "1",
        "gold_answer_notes": (
            "Participants were offered up to six one-to-one sessions "
            "with LISTEN-trained practitioners together with a "
            "co-designed digital resource and paper-based book."
        ),
        "verification_notes": (
            "Manually verified from the LISTEN abstract on page 1."
        ),
    },
    {
        "question_id": "EVAL_028",
        "question": (
            "What methods were planned for the LISTEN "
            "process evaluation?"
        ),
        "gold_chunk_ids": "PROTO_004_P003_C001",
        "gold_pages": "3",
        "gold_answer_notes": (
            "The process evaluation involved implementation scales, "
            "interviews, and focus groups."
        ),
        "verification_notes": (
            "Manually verified from LISTEN page 3."
        ),
    },
    {
        "question_id": "EVAL_029",
        "question": (
            "Did LISTEN participants have to personally speak English "
            "or Welsh to be eligible?"
        ),
        "gold_chunk_ids": "PROTO_004_P003_C001",
        "gold_pages": "3",
        "gold_answer_notes": (
            "No. Participants could be English or Welsh speakers, "
            "or they could have access to someone able to act "
            "as a translator."
        ),
        "verification_notes": (
            "Manually verified from LISTEN eligibility criteria "
            "on page 3."
        ),
    },
    {
        "question_id": "EVAL_030",
        "question": (
            "According to the LISTEN schedule of events, which "
            "participants completed the process-evaluation questionnaires?"
        ),
        "gold_chunk_ids": (
            "PROTO_004_P004_C001;"
            "PROTO_004_P004_C002"
        ),
        "gold_pages": "4",
        "gold_answer_notes": (
            "Process-evaluation questionnaires were completed only "
            "by participants in the LISTEN intervention group."
        ),
        "verification_notes": (
            "Manually verified from the SPIRIT schedule notes "
            "on LISTEN page 4."
        ),
    },
]


# Add the verified LISTEN questions to the evaluation dataframe
for record in listen_supported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display the completed supported LISTEN questions
print(
    evaluation_df.loc[
        evaluation_df["question_id"].isin(
            [f"EVAL_{number:03d}" for number in range(25, 31)]
        ),
        [
            "question_id",
            "question_type",
            "question",
            "gold_chunk_ids",
            "gold_pages",
        ],
    ].to_string(index=False)
)

question_id   question_type                                                                                                        question                                              gold_chunk_ids gold_pages
   EVAL_025 straightforward                            At what time points were LISTEN participants asked to complete study questionnaires?                     PROTO_004_P001_C002;PROTO_004_P003_C001        1;3
   EVAL_026 straightforward                                How could eligible participants be referred or self-refer into the LISTEN trial? PROTO_004_P001_C002;PROTO_004_P002_C003;PROTO_004_P003_C002      1;2;3
   EVAL_027       difficult                    What intervention support was offered to participants randomised to the LISTEN intervention?                                         PROTO_004_P001_C002          1
   EVAL_028       difficult                                                    What methods were planned for the LISTEN process evaluation?                 

## 33. Verify unsupported LISTEN questions

The unsupported LISTEN questions will focus on completed effectiveness and economic results.

The protocol describes how effectiveness and cost-effectiveness will be evaluated, but these questions will only be labelled unsupported if the document does not report completed trial findings.

In [35]:
# Search for wording that could indicate completed LISTEN results
listen_result_terms = [
    "results showed",
    "significantly improved",
    "significant improvement",
    "was effective",
    "was cost-effective",
    "incremental cost",
    "final results",
    "participants improved",
]


for term in listen_result_terms:

    matches = listen_chunks[
        listen_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(
        f"{term}: {len(matches)} matching chunks"
    )

results showed: 0 matching chunks
significantly improved: 0 matching chunks
significant improvement: 0 matching chunks
was effective: 0 matching chunks
was cost-effective: 0 matching chunks
incremental cost: 0 matching chunks
final results: 0 matching chunks
participants improved: 0 matching chunks


## 34. Add the unsupported LISTEN evaluation questions

Two LISTEN questions were verified as unsupported.

The protocol describes planned effectiveness and cost-effectiveness analyses, but it does not report completed LISTEN trial results for either outcome.

The expected response for both questions is therefore `Insufficient Evidence`.

In [36]:
# Define the two verified unsupported LISTEN questions
listen_unsupported_questions = [
    {
        "question_id": "EVAL_031",
        "question": (
            "Did the LISTEN intervention ultimately improve participation "
            "compared with usual care?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol describes planned effectiveness evaluation "
            "but does not report completed LISTEN trial results."
        ),
    },
    {
        "question_id": "EVAL_032",
        "question": (
            "What were the final cost-effectiveness results of LISTEN?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol describes planned economic evaluation "
            "but does not report final LISTEN cost-effectiveness results."
        ),
    },
]


# Add the unsupported questions to the evaluation dataframe
for record in listen_unsupported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display all eight LISTEN evaluation questions
listen_eval = evaluation_df[
    evaluation_df["document_name"] == "LISTEN"
].copy()


print(
    listen_eval[
        [
            "question_id",
            "question_type",
            "expected_support",
            "question",
            "gold_chunk_ids",
            "gold_pages",
            "gold_answer_notes",
        ]
    ].to_string(index=False)
)

question_id   question_type expected_support                                                                                                        question                                              gold_chunk_ids gold_pages                                                                                                                                            gold_answer_notes
   EVAL_025 straightforward        supported                            At what time points were LISTEN participants asked to complete study questionnaires?                     PROTO_004_P001_C002;PROTO_004_P003_C001        1;3                                                          Questionnaires were completed at baseline and repeated at 6 weeks and 3 months after randomisation.
   EVAL_026 straightforward        supported                                How could eligible participants be referred or self-refer into the LISTEN trial? PROTO_004_P001_C002;PROTO_004_P002_C003;PROTO_004_P003_C002      1;2;3   

## 35. Identify THP_TA evaluation evidence

THP_TA is the final protocol in the held-out evaluation set.

The exclusion-criteria and sample-size questions used during Day 3 development will not be reused. We will inspect several other parts of the protocol and select six supported questions from clearly verified evidence.

In [38]:
# Keep only THP_TA chunks
thp_ta_chunks = (
    chunks_df[
        chunks_df["document_name"] == "THP_TA"
    ]
    .copy()
    .reset_index(drop=True)
)


# Search across several different protocol topics
thp_ta_candidate_terms = {
    "randomisation": "random",
    "primary_outcome": "primary outcome",
    "intervention": "intervention",
    "control": "control",
    "follow_up": "follow",
    "blinding": "blind",
    "adverse_events": "adverse",
    "questionnaire": "questionnaire",
}


for topic, phrase in thp_ta_candidate_terms.items():

    matches = thp_ta_chunks[
        thp_ta_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    # Show only two compact passages per topic
    for _, row in matches.head(2).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 250,
        )

        end = min(
            len(text),
            position + len(phrase) + 500,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: randomisation
Matching chunks: 16

PROTO_005_P001_C001 | Page 1
stated in a credit line to the data. Trials Technology-assisted cognitive-behavior  therapy delivered by peers versus standard  cognitive behavior therapy delivered  by community health workers for perinatal  depression: study protocol of a cluster  randomized controlled non-inferiority trial Atif Rahman1*  , Abid Malik2, Najia Atif3, Huma Nazir3, Ahmed Zaidi3, Anum Nisar3, Ahmed Waqas1,  Maria Sharif3, Tao Chen4, Duolao Wang5 and Siham Sikander1  Abstract  Background The lack of trained mental health professionals is a key barrier to scale-up of evidence-based psycho- logical interventions in low and middle-income countries. We have developed an app that allows a peer with no prior  experience of health-care delivery to deliver the cognitive the

PROTO_005_P001_C002 | Page 1
ssess the effectiveness and cost-effectiveness of this Tech- nology-assisted peer-delivered THP versus standard face-to-face Thinking Healthy

## 36. Verify specific THP_TA evidence

The initial THP_TA search identified several strong evaluation topics.

We will now inspect six specific facts in compact passages so that the exact supporting chunk IDs can be recorded before creating the final supported questions.

In [39]:
# Search for six specific THP_TA facts
thp_ta_gold_candidates = {
    "delivery_comparison": "peer-delivered",
    "cluster_randomisation": "70 village clusters",
    "primary_outcome": "remission from major depressive episode",
    "blinding": "not to disclose",
    "phq9": "Patient health questionnaire",
    "adverse_events": "detecting the adverse events",
}


for topic, phrase in thp_ta_gold_candidates.items():

    matches = thp_ta_chunks[
        thp_ta_chunks["text"].str.contains(
            phrase,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTOPIC: {topic}")
    print("Matching chunks:", len(matches))

    # Keep the output small
    for _, row in matches.head(2).iterrows():

        text = row["text"]

        position = text.lower().find(
            phrase.lower()
        )

        start = max(
            0,
            position - 300,
        )

        end = min(
            len(text),
            position + len(phrase) + 700,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TOPIC: delivery_comparison
Matching chunks: 7

PROTO_005_P001_C002 | Page 1
developed an app that allows a peer with no prior  experience of health-care delivery to deliver the cognitive therapy-based intervention for perinatal depression,  the Thinking Healthy Programme (THP). This trial aims to assess the effectiveness and cost-effectiveness of this Tech- nology-assisted peer-delivered THP versus standard face-to-face Thinking Healthy Programme delivered by trained  health workers. Methods We will employ a non-inferiority stratified cluster randomized controlled trial design comparing the two  formats of intervention delivery. A total of 980 women in the second or third trimester of pregnancy with a diag- nosis of Major Depressive Episode, evaluated with the Structured Clinical Interview for DSM-V Disorders (SCID),  will be recruited into the trial. The unit of randomization will be 70 village clusters randomly allocated in a 1:1 ratio  to the intervention and control arms. The prim

## 37. Add the supported THP_TA evaluation questions

Six supported THP_TA questions were created from manually verified evidence.

The exclusion-criteria and sample-size questions used during Day 3 development are excluded. The new questions cover intervention delivery, randomisation, outcome measurement, masking, secondary outcomes, and adverse-event monitoring./

In [40]:
# Define the six manually verified THP_TA questions
thp_ta_supported_questions = [
    {
        "question_id": "EVAL_033",
        "question": (
            "What two approaches to delivering the Thinking Healthy "
            "Programme were compared in the THP_TA trial?"
        ),
        "gold_chunk_ids": "PROTO_005_P001_C002",
        "gold_pages": "1",
        "gold_answer_notes": (
            "The trial compared technology-assisted THP delivered by "
            "peers with standard face-to-face THP delivered by trained "
            "health workers."
        ),
        "verification_notes": (
            "Manually verified from THP_TA page 1."
        ),
    },
    {
        "question_id": "EVAL_034",
        "question": (
            "What was the unit of randomisation in THP_TA and how "
            "were clusters allocated between the trial arms?"
        ),
        "gold_chunk_ids": (
            "PROTO_005_P001_C002;"
            "PROTO_005_P003_C002"
        ),
        "gold_pages": "1;3",
        "gold_answer_notes": (
            "The village cluster was the unit of randomisation. "
            "Seventy village clusters were allocated in a 1:1 ratio "
            "to the intervention and control arms."
        ),
        "verification_notes": (
            "Manually verified from THP_TA pages 1 and 3."
        ),
    },
    {
        "question_id": "EVAL_035",
        "question": (
            "How was the primary outcome of THP_TA defined, "
            "when was it assessed, and which instrument was used?"
        ),
        "gold_chunk_ids": "PROTO_005_P001_C002",
        "gold_pages": "1",
        "gold_answer_notes": (
            "The primary outcome was remission from major depressive "
            "episode at 3 months postnatal, measured using the SCID."
        ),
        "verification_notes": (
            "Manually verified from THP_TA page 1."
        ),
    },
    {
        "question_id": "EVAL_036",
        "question": (
            "How did THP_TA reduce the risk that outcome assessors "
            "would learn participants' treatment allocation?"
        ),
        "gold_chunk_ids": "PROTO_005_P006_C002",
        "gold_pages": "6",
        "gold_answer_notes": (
            "Outcome assessors were masked to treatment allocation, "
            "participants were instructed not to disclose how they "
            "received the intervention, and any unmasking was recorded."
        ),
        "verification_notes": (
            "Manually verified from THP_TA page 6."
        ),
    },
    {
        "question_id": "EVAL_037",
        "question": (
            "Was the PHQ-9 used as the THP_TA primary outcome measure, "
            "or what was it used to assess?"
        ),
        "gold_chunk_ids": "PROTO_005_P007_C003",
        "gold_pages": "7",
        "gold_answer_notes": (
            "No. The PHQ-9 was used as a secondary outcome measure "
            "of depressive symptoms, assessed at 3 and 6 months postnatal."
        ),
        "verification_notes": (
            "Manually verified from THP_TA page 7."
        ),
    },
    {
        "question_id": "EVAL_038",
        "question": (
            "Who was responsible for detecting adverse events in "
            "THP_TA, and at what stages were they monitored?"
        ),
        "gold_chunk_ids": "PROTO_005_P005_C002",
        "gold_pages": "5",
        "gold_answer_notes": (
            "The outcome assessment team monitored adverse events at "
            "3 and 6 months postnatal, while LHWs or peers monitored "
            "them throughout intervention delivery in both arms."
        ),
        "verification_notes": (
            "Manually verified from THP_TA page 5."
        ),
    },
]


# Add the verified THP_TA questions to the evaluation dataframe
for record in thp_ta_supported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display the six supported THP_TA questions
print(
    evaluation_df.loc[
        evaluation_df["question_id"].isin(
            [f"EVAL_{number:03d}" for number in range(33, 39)]
        ),
        [
            "question_id",
            "question_type",
            "question",
            "gold_chunk_ids",
            "gold_pages",
        ],
    ].to_string(index=False)
)

question_id   question_type                                                                                              question                          gold_chunk_ids gold_pages
   EVAL_033 straightforward   What two approaches to delivering the Thinking Healthy Programme were compared in the THP_TA trial?                     PROTO_005_P001_C002          1
   EVAL_034 straightforward  What was the unit of randomisation in THP_TA and how were clusters allocated between the trial arms? PROTO_005_P001_C002;PROTO_005_P003_C002        1;3
   EVAL_035       difficult   How was the primary outcome of THP_TA defined, when was it assessed, and which instrument was used?                     PROTO_005_P001_C002          1
   EVAL_036       difficult How did THP_TA reduce the risk that outcome assessors would learn participants' treatment allocation?                     PROTO_005_P006_C002          6
   EVAL_037      confusable              Was the PHQ-9 used as the THP_TA primary outcome measu

## 38. Verify unsupported THP_TA questions

The final two evaluation questions will ask about completed THP_TA effectiveness and cost-effectiveness results.

The protocol describes how these outcomes will be evaluated, but the questions will only be labelled unsupported if the document does not report completed trial findings.

In [41]:
# Search for wording that could indicate completed THP_TA trial results
thp_ta_result_terms = [
    "results showed",
    "was non-inferior",
    "non-inferiority was",
    "remission rate was",
    "significantly improved",
    "was effective",
    "was cost-effective",
    "incremental cost",
    "final results",
]


for term in thp_ta_result_terms:

    matches = thp_ta_chunks[
        thp_ta_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(
        f"{term}: {len(matches)} matching chunks"
    )

results showed: 0 matching chunks
was non-inferior: 0 matching chunks
non-inferiority was: 0 matching chunks
remission rate was: 0 matching chunks
significantly improved: 0 matching chunks
was effective: 1 matching chunks
was cost-effective: 0 matching chunks
incremental cost: 1 matching chunks
final results: 2 matching chunks


## 39. Inspect possible THP_TA result-like passages

A few result-like phrases were found in the THP_TA protocol.

Before labelling the final two questions as unsupported, we will inspect these passages to determine whether they report completed THP_TA trial findings, previous-study evidence, or planned analyses.

In [42]:
# Inspect only the result-like phrases that produced matches
result_like_terms = [
    "was effective",
    "incremental cost",
    "final results",
]


for term in result_like_terms:

    matches = thp_ta_chunks[
        thp_ta_chunks["text"].str.contains(
            term,
            case=False,
            na=False,
            regex=False,
        )
    ]

    print(f"\nTERM: {term}")

    for _, row in matches.iterrows():

        text = row["text"]

        position = text.lower().find(
            term.lower()
        )

        start = max(
            0,
            position - 350,
        )

        end = min(
            len(text),
            position + len(term) + 750,
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        print()
        print(
            f"{row['chunk_id']} | Page {row['page_number']}"
        )
        print(snippet)


TERM: was effective

PROTO_005_P002_C002 | Page 2
ship with the infant and significant others and inter-session practice activities to help  the mother and family to problem-solve [8]. A large randomized controlled trial showed that THP more than  halved the rate of depression compared with usual care  and led to significant improvements in women’s functioning and disability, and the intervention was effective  in the poorest populations [9, 10]. In 2015, the THP was  incorporated into the World Health Organization’s flagship Mental Health Gap Action Programme (mhGAP)  for global dissemination [11]. A key barrier to scale-up of the THP in LMICs is the  lack of trained health professionals to deliver the intervention. Even where health workers are available, they  are over-burdened, which makes it difficult to sustain  the program beyond pilot sites. In recent studies, peers  (women from the same localities with no prior experience of health-care delivery) have been found to be feasibl

## 40. Add the unsupported THP_TA evaluation questions

Two THP_TA questions were verified as unsupported.

Result-like wording in the protocol was inspected manually. The apparent matches referred to an earlier THP trial, planned cost-effectiveness analysis, or an external census reference rather than completed THP_TA findings.

The expected response for both questions is therefore `Insufficient Evidence`.

In [43]:
# Define the two verified unsupported THP_TA questions
thp_ta_unsupported_questions = [
    {
        "question_id": "EVAL_039",
        "question": (
            "Did technology-assisted peer-delivered THP ultimately "
            "demonstrate non-inferior depression remission compared "
            "with standard THP?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol describes the planned non-inferiority trial "
            "and cites results from earlier THP studies, but it does not "
            "report completed THP_TA non-inferiority results."
        ),
    },
    {
        "question_id": "EVAL_040",
        "question": (
            "What were the final cost-effectiveness results of THP_TA?"
        ),
        "gold_chunk_ids": "",
        "gold_pages": "",
        "gold_answer_notes": "Insufficient Evidence",
        "verification_notes": (
            "The protocol describes planned cost-effectiveness analysis "
            "and ICER estimation but does not report completed THP_TA "
            "economic results."
        ),
    },
]


# Add the unsupported questions to the evaluation dataframe
for record in thp_ta_unsupported_questions:

    mask = (
        evaluation_df["question_id"]
        == record["question_id"]
    )

    for column, value in record.items():

        if column != "question_id":
            evaluation_df.loc[
                mask,
                column,
            ] = value


# Display all eight THP_TA questions
thp_ta_eval = evaluation_df[
    evaluation_df["document_name"] == "THP_TA"
].copy()


print(
    thp_ta_eval[
        [
            "question_id",
            "question_type",
            "expected_support",
            "question",
            "gold_chunk_ids",
            "gold_pages",
            "gold_answer_notes",
        ]
    ].to_string(index=False)
)

question_id   question_type expected_support                                                                                                                        question                          gold_chunk_ids gold_pages                                                                                                                                                       gold_answer_notes
   EVAL_033 straightforward        supported                             What two approaches to delivering the Thinking Healthy Programme were compared in the THP_TA trial?                     PROTO_005_P001_C002          1                                       The trial compared technology-assisted THP delivered by peers with standard face-to-face THP delivered by trained health workers.
   EVAL_034 straightforward        supported                            What was the unit of randomisation in THP_TA and how were clusters allocated between the trial arms? PROTO_005_P001_C002;PROTO_005_P003_C002    

## 41. Validate and save the gold evaluation set

The manually verified evaluation dataset is now complete.

Before running the RAG system, we will validate its structure to make sure that all 40 questions are complete, supported questions contain gold evidence, unsupported questions contain no gold evidence, question IDs are unique, and the expected question distribution is preserved.

The validated dataset will then be saved so that the same fixed gold set is used for all subsequent retrieval and generation experiments.

In [45]:
from pathlib import Path

# Define the root folder for this project
project_root = Path(
    "/Users/arvindshine/Clinical Trial Protocol Intelligence Copilot"
)

print(project_root)

/Users/arvindshine/Clinical Trial Protocol Intelligence Copilot


In [46]:
# Work with the evaluation questions in ID order
gold_eval_df = (
    evaluation_df
    .sort_values("question_id")
    .reset_index(drop=True)
    .copy()
)


# Check the expected total number of questions
assert len(gold_eval_df) == 40, (
    f"Expected 40 questions, found {len(gold_eval_df)}."
)


# Check that question IDs are unique
assert gold_eval_df["question_id"].is_unique, (
    "Duplicate question IDs found."
)


# Check that question text is present and unique
assert gold_eval_df["question"].notna().all(), (
    "Some questions are missing."
)

assert (
    gold_eval_df["question"].str.strip() != ""
).all(), "Some questions are blank."

assert gold_eval_df["question"].is_unique, (
    "Duplicate evaluation questions found."
)


# Split supported and unsupported questions
supported_eval = gold_eval_df[
    gold_eval_df["expected_support"] == "supported"
].copy()

unsupported_eval = gold_eval_df[
    gold_eval_df["expected_support"] == "unsupported"
].copy()


# Supported questions must contain gold evidence
assert (
    supported_eval["gold_chunk_ids"]
    .fillna("")
    .str.strip()
    .ne("")
    .all()
), "Some supported questions are missing gold chunk IDs."

assert (
    supported_eval["gold_pages"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), "Some supported questions are missing gold pages."


# Unsupported questions must not contain gold evidence
assert (
    unsupported_eval["gold_chunk_ids"]
    .fillna("")
    .str.strip()
    .eq("")
    .all()
), "Some unsupported questions incorrectly contain gold chunk IDs."

assert (
    unsupported_eval["gold_pages"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .all()
), "Some unsupported questions incorrectly contain gold pages."


# Unsupported questions should expect the exact abstention response
assert (
    unsupported_eval["gold_answer_notes"]
    == "Insufficient Evidence"
).all(), "Unsupported questions have inconsistent gold answers."


# Check the intended supported/unsupported balance
assert len(supported_eval) == 30, (
    f"Expected 30 supported questions, found {len(supported_eval)}."
)

assert len(unsupported_eval) == 10, (
    f"Expected 10 unsupported questions, found {len(unsupported_eval)}."
)


# Check that every protocol has exactly eight questions
questions_per_protocol = (
    gold_eval_df
    .groupby("document_name")
    .size()
)

assert (
    questions_per_protocol == 8
).all(), "Each protocol should have exactly 8 questions."


# Check the intended question-type distribution
question_type_counts = (
    gold_eval_df["question_type"]
    .value_counts()
    .sort_index()
)


# Save the locked gold-standard evaluation dataset
gold_eval_path = (
    project_root
    / "data"
    / "metadata"
    / "rag_evaluation_gold.csv"
)

gold_eval_df.to_csv(
    gold_eval_path,
    index=False,
)


print("Gold evaluation validation passed.")
print()
print("Total questions:", len(gold_eval_df))
print("Supported:", len(supported_eval))
print("Unsupported:", len(unsupported_eval))

print("\nQuestions per protocol:")
print(questions_per_protocol.to_string())

print("\nQuestion types:")
print(question_type_counts.to_string())

print("\nSaved to:")
print(gold_eval_path)

Gold evaluation validation passed.

Total questions: 40
Supported: 30
Unsupported: 10

Questions per protocol:
document_name
CARE_STROKE    8
CERTAIN        8
INTEGRA        8
LISTEN         8
THP_TA         8

Question types:
question_type
confusable          5
difficult          10
straightforward    10
structure_heavy     5
unsupported        10

Saved to:
/Users/arvindshine/Clinical Trial Protocol Intelligence Copilot/data/metadata/rag_evaluation_gold.csv


## 42. Set up the baseline hybrid retriever

The gold evaluation set is now fixed.

We will evaluate the existing retrieval pipeline without changing its configuration. The baseline combines OpenAI semantic retrieval with BM25 lexical retrieval using Reciprocal Rank Fusion.

Each question is searched only within its known protocol because the final application is designed as document-scoped protocol question answering.

Improvements will only be considered after the baseline failures have been measured.

In [47]:
import re
import chromadb

from openai import OpenAI
from rank_bm25 import BM25Okapi


# Use the same retrieval settings selected on Day 4
embedding_model_name = "text-embedding-3-small"

semantic_candidate_k = 10
bm25_candidate_k = 10
evaluation_top_k = 10
rrf_k = 60


# Connect to OpenAI
openai_client = OpenAI()


# Open the persistent Chroma database created on Day 3
chroma_client = chromadb.PersistentClient(
    path=str(project_root / "data" / "chroma")
)


# Reuse the existing OpenAI embedding collection
openai_collection = chroma_client.get_collection(
    name="protocol_chunks_openai_v1"
)


def retrieval_tokenize(text):
    """
    Simple tokenizer used by the BM25 baseline.
    """

    return re.findall(
        r"[A-Za-z0-9]+",
        text.lower(),
    )


def hybrid_retrieve_for_evaluation(
    question,
    document_name,
    semantic_k=semantic_candidate_k,
    bm25_k=bm25_candidate_k,
    top_k=evaluation_top_k,
):
    """
    Run the frozen Day 4 hybrid retrieval pipeline.

    Semantic search and BM25 are performed within the selected protocol,
    and their rankings are combined using Reciprocal Rank Fusion.
    """

    # Keep BM25 search inside the requested protocol
    document_chunks = (
        chunks_df[
            chunks_df["document_name"] == document_name
        ]
        .copy()
        .reset_index(drop=True)
    )


    # Embed the evaluation question
    embedding_response = openai_client.embeddings.create(
        model=embedding_model_name,
        input=question,
    )

    query_embedding = (
        embedding_response
        .data[0]
        .embedding
    )


    # Retrieve semantic candidates from the same protocol
    semantic_results = openai_collection.query(
        query_embeddings=[query_embedding],
        n_results=min(
            semantic_k,
            len(document_chunks),
        ),
        where={
            "document_name": document_name
        },
        include=[
            "documents",
            "metadatas",
            "distances",
        ],
    )


    # Convert semantic results into ranks
    semantic_ranks = {}

    for rank, chunk_id in enumerate(
        semantic_results["ids"][0],
        start=1,
    ):
        semantic_ranks[chunk_id] = rank


    # Build BM25 over the selected protocol
    tokenized_chunks = [
        retrieval_tokenize(text)
        for text in document_chunks["text"]
    ]

    bm25 = BM25Okapi(
        tokenized_chunks
    )

    query_tokens = retrieval_tokenize(
        question
    )

    bm25_scores = bm25.get_scores(
        query_tokens
    )


    # Rank BM25 candidates from highest score to lowest
    bm25_order = (
        bm25_scores
        .argsort()[::-1]
        [:min(bm25_k, len(document_chunks))]
    )


    bm25_ranks = {}

    for rank, row_index in enumerate(
        bm25_order,
        start=1,
    ):

        chunk_id = document_chunks.loc[
            row_index,
            "chunk_id",
        ]

        bm25_ranks[chunk_id] = rank


    # Combine all candidates returned by either retriever
    candidate_ids = (
        set(semantic_ranks)
        | set(bm25_ranks)
    )


    fused_rows = []


    for chunk_id in candidate_ids:

        semantic_rank = semantic_ranks.get(
            chunk_id
        )

        bm25_rank = bm25_ranks.get(
            chunk_id
        )


        # Reciprocal Rank Fusion score
        rrf_score = 0.0

        if semantic_rank is not None:
            rrf_score += (
                1 / (rrf_k + semantic_rank)
            )

        if bm25_rank is not None:
            rrf_score += (
                1 / (rrf_k + bm25_rank)
            )


        chunk_row = document_chunks[
            document_chunks["chunk_id"]
            == chunk_id
        ].iloc[0]


        fused_rows.append(
            {
                "chunk_id": chunk_id,
                "document_name": document_name,
                "page_number": chunk_row["page_number"],
                "semantic_rank": semantic_rank,
                "bm25_rank": bm25_rank,
                "rrf_score": rrf_score,
            }
        )


    # Highest RRF score becomes rank 1
    ranked_df = (
        pd.DataFrame(fused_rows)
        .sort_values(
            "rrf_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )


    ranked_df["hybrid_rank"] = (
        ranked_df.index + 1
    )


    return ranked_df.head(
        top_k
    )

In [48]:
test_retrieval = hybrid_retrieve_for_evaluation(
    question=gold_eval_df.loc[0, "question"],
    document_name=gold_eval_df.loc[0, "document_name"],
)

print(
    test_retrieval[
        [
            "hybrid_rank",
            "chunk_id",
            "page_number",
            "semantic_rank",
            "bm25_rank",
            "rrf_score",
        ]
    ].to_string(index=False)
)

 hybrid_rank            chunk_id  page_number  semantic_rank  bm25_rank  rrf_score
           1 PROTO_001_P003_C002            3            1.0        3.0   0.032266
           2 PROTO_001_P003_C001            3            3.0        2.0   0.032002
           3 PROTO_001_P004_C003            4            2.0        6.0   0.031281
           4 PROTO_001_P004_C001            4            4.0        4.0   0.031250
           5 PROTO_001_P002_C004            2            5.0        5.0   0.030769
           6 PROTO_001_P003_C003            3            6.0       10.0   0.029437
           7 PROTO_001_P005_C003            5           10.0        8.0   0.028992
           8 PROTO_001_P001_C002            1            NaN        1.0   0.016393
           9 PROTO_001_P006_C001            6            NaN        7.0   0.014925
          10 PROTO_001_P005_C001            5            7.0        NaN   0.014925


## 43. Evaluate baseline retrieval

The first held-out smoke test revealed a useful failure case: BM25 ranked the correct evidence first, but the hybrid RRF ranking moved it down because it was absent from the semantic top 10.

We will now run the unchanged baseline retriever across all 30 supported questions.

For each question, we record the best rank of any acceptable gold chunk and calculate Hit@1, Hit@3, Hit@5, and Hit@10. No retrieval settings will be changed until these baseline results are complete.

In [49]:
retrieval_evaluation_rows = []


for _, row in supported_eval.iterrows():

    # Run the frozen baseline retriever
    retrieved_df = hybrid_retrieve_for_evaluation(
        question=row["question"],
        document_name=row["document_name"],
        top_k=10,
    )


    # Multiple semicolon-separated chunks may be acceptable gold evidence
    gold_chunk_ids = [
        chunk_id.strip()
        for chunk_id in row["gold_chunk_ids"].split(";")
        if chunk_id.strip()
    ]


    # Keep retrieved IDs in ranked order
    retrieved_chunk_ids = (
        retrieved_df["chunk_id"]
        .tolist()
    )


    # Find the rank of every acceptable gold chunk that was retrieved
    gold_ranks = [
        retrieved_chunk_ids.index(chunk_id) + 1
        for chunk_id in gold_chunk_ids
        if chunk_id in retrieved_chunk_ids
    ]


    # Best acceptable evidence rank
    best_gold_rank = (
        min(gold_ranks)
        if gold_ranks
        else None
    )


    retrieval_evaluation_rows.append(
        {
            "question_id": row["question_id"],
            "document_name": row["document_name"],
            "question_type": row["question_type"],
            "question": row["question"],
            "gold_chunk_ids": row["gold_chunk_ids"],
            "best_gold_rank": best_gold_rank,
            "hit_at_1": (
                best_gold_rank is not None
                and best_gold_rank <= 1
            ),
            "hit_at_3": (
                best_gold_rank is not None
                and best_gold_rank <= 3
            ),
            "hit_at_5": (
                best_gold_rank is not None
                and best_gold_rank <= 5
            ),
            "hit_at_10": (
                best_gold_rank is not None
                and best_gold_rank <= 10
            ),
            "retrieved_top_10": ";".join(
                retrieved_chunk_ids
            ),
        }
    )


retrieval_eval_df = pd.DataFrame(
    retrieval_evaluation_rows
)


# Calculate aggregate retrieval metrics
retrieval_metrics = {
    "Hit@1": retrieval_eval_df["hit_at_1"].mean(),
    "Hit@3": retrieval_eval_df["hit_at_3"].mean(),
    "Hit@5": retrieval_eval_df["hit_at_5"].mean(),
    "Hit@10": retrieval_eval_df["hit_at_10"].mean(),
}


print("Baseline retrieval metrics")
print()

for metric, value in retrieval_metrics.items():
    print(
        f"{metric}: {value:.3f} "
        f"({int(value * len(retrieval_eval_df))}/{len(retrieval_eval_df)})"
    )


# Show questions where the gold evidence missed the final Top 5
top5_failures = retrieval_eval_df[
    ~retrieval_eval_df["hit_at_5"]
].copy()


print()
print(
    "Questions missing gold evidence from Top 5:",
    len(top5_failures),
)


if len(top5_failures) > 0:

    print()

    print(
        top5_failures[
            [
                "question_id",
                "document_name",
                "question_type",
                "best_gold_rank",
                "question",
            ]
        ].to_string(index=False)
    ) 

Baseline retrieval metrics

Hit@1: 0.600 (18/30)
Hit@3: 0.733 (22/30)
Hit@5: 0.800 (24/30)
Hit@10: 0.933 (28/30)

Questions missing gold evidence from Top 5: 6

question_id document_name   question_type  best_gold_rank                                                                                            question
   EVAL_001       CERTAIN straightforward             8.0       How long were participants in the CERTAIN study followed after the first counselling session?
   EVAL_002       CERTAIN straightforward             9.0    What personal technology were smokeless tobacco users required to have for inclusion in CERTAIN?
   EVAL_019       INTEGRA       difficult             6.0                  How was the INTEGRA control group selected and matched to the intervention groups?
   EVAL_027        LISTEN       difficult             6.0        What intervention support was offered to participants randomised to the LISTEN intervention?
   EVAL_028        LISTEN       difficult        

## 44. Diagnose baseline retrieval failures

The baseline retrieved acceptable evidence within the top 5 for 24 of 30 supported questions.

For the six remaining questions, we will determine whether the correct evidence was retrieved by either semantic search or BM25 but ranked too low by RRF, or whether both candidate retrievers missed the evidence entirely.

This distinction tells us whether a reranking improvement is likely to help or whether the candidate retrieval stage itself needs improvement.

In [50]:
failure_diagnostics = []


for _, row in top5_failures.iterrows():

    # Retrieve the full union of the semantic Top 10 and BM25 Top 10
    # The union can contain at most 20 unique chunks.
    diagnostic_results = hybrid_retrieve_for_evaluation(
        question=row["question"],
        document_name=row["document_name"],
        semantic_k=10,
        bm25_k=10,
        top_k=20,
    )


    # Parse all acceptable gold chunks for this question
    gold_ids = [
        chunk_id.strip()
        for chunk_id in row["gold_chunk_ids"].split(";")
        if chunk_id.strip()
    ]


    # Find any acceptable gold chunks present in the candidate union
    gold_matches = diagnostic_results[
        diagnostic_results["chunk_id"].isin(
            gold_ids
        )
    ].copy()


    if gold_matches.empty:

        # Neither semantic Top 10 nor BM25 Top 10 found acceptable evidence
        failure_type = "CANDIDATE_RECALL_MISS"

        failure_diagnostics.append(
            {
                "question_id": row["question_id"],
                "document_name": row["document_name"],
                "question_type": row["question_type"],
                "failure_type": failure_type,
                "gold_chunk_id": None,
                "semantic_rank": None,
                "bm25_rank": None,
                "hybrid_rank": None,
            }
        )

    else:

        # Use the highest-ranked acceptable gold chunk
        best_gold = (
            gold_matches
            .sort_values("hybrid_rank")
            .iloc[0]
        )

        failure_type = "RANKING_MISS"

        failure_diagnostics.append(
            {
                "question_id": row["question_id"],
                "document_name": row["document_name"],
                "question_type": row["question_type"],
                "failure_type": failure_type,
                "gold_chunk_id": best_gold["chunk_id"],
                "semantic_rank": best_gold["semantic_rank"],
                "bm25_rank": best_gold["bm25_rank"],
                "hybrid_rank": best_gold["hybrid_rank"],
            }
        )


failure_diagnostics_df = pd.DataFrame(
    failure_diagnostics
)


print(
    failure_diagnostics_df.to_string(
        index=False
    )
)


print("\nFailure-type counts:")

print(
    failure_diagnostics_df[
        "failure_type"
    ]
    .value_counts()
    .to_string()
)

question_id document_name   question_type          failure_type       gold_chunk_id  semantic_rank  bm25_rank  hybrid_rank
   EVAL_001       CERTAIN straightforward          RANKING_MISS PROTO_001_P001_C002            NaN        1.0          8.0
   EVAL_002       CERTAIN straightforward          RANKING_MISS PROTO_001_P001_C004            NaN        5.0          9.0
   EVAL_019       INTEGRA       difficult          RANKING_MISS PROTO_003_P005_C002            6.0        7.0          6.0
   EVAL_027        LISTEN       difficult          RANKING_MISS PROTO_004_P001_C002            NaN        1.0          6.0
   EVAL_028        LISTEN       difficult          RANKING_MISS PROTO_004_P003_C001            NaN        9.0         13.0
   EVAL_035        THP_TA       difficult CANDIDATE_RECALL_MISS                 NaN            NaN        NaN          NaN

Failure-type counts:
failure_type
RANKING_MISS             5
CANDIDATE_RECALL_MISS    1


## 45. Diagnose the remaining candidate-recall miss

Five of the six Top-5 failures are ranking problems, which suggests that reranking may improve the system.

One question, EVAL_035, is different because its gold evidence was absent from both the semantic and BM25 Top 10 candidate sets.

Before adding a reranker, we will test whether slightly deeper candidate retrieval can recover this evidence. The frozen baseline metrics will remain unchanged.

In [51]:
# Inspect the one true candidate-recall failure
eval_035 = gold_eval_df[
    gold_eval_df["question_id"] == "EVAL_035"
].iloc[0]


gold_ids = [
    chunk_id.strip()
    for chunk_id in eval_035["gold_chunk_ids"].split(";")
    if chunk_id.strip()
]


diagnostic_rows = []


# Test progressively deeper candidate pools
for candidate_k in [10, 20, 30]:

    results = hybrid_retrieve_for_evaluation(
        question=eval_035["question"],
        document_name=eval_035["document_name"],
        semantic_k=candidate_k,
        bm25_k=candidate_k,
        top_k=60,
    )


    gold_matches = results[
        results["chunk_id"].isin(gold_ids)
    ]


    if gold_matches.empty:

        diagnostic_rows.append(
            {
                "candidate_k": candidate_k,
                "gold_found": False,
                "semantic_rank": None,
                "bm25_rank": None,
                "hybrid_rank": None,
            }
        )

    else:

        best_match = (
            gold_matches
            .sort_values("hybrid_rank")
            .iloc[0]
        )

        diagnostic_rows.append(
            {
                "candidate_k": candidate_k,
                "gold_found": True,
                "semantic_rank": best_match["semantic_rank"],
                "bm25_rank": best_match["bm25_rank"],
                "hybrid_rank": best_match["hybrid_rank"],
            }
        )


candidate_depth_df = pd.DataFrame(
    diagnostic_rows
)


print(candidate_depth_df.to_string(index=False))

 candidate_k  gold_found  semantic_rank  bm25_rank  hybrid_rank
          10       False            NaN        NaN          NaN
          20        True            NaN       12.0         19.0
          30        True           23.0       12.0         17.0


## 46. Add a cross-encoder reranker

The baseline analysis showed that most retrieval failures were ranking failures rather than complete evidence misses.

We will therefore test a two-stage retrieval approach. Semantic search and BM25 will first create a wider candidate pool, and a cross-encoder reranker will then score each question–chunk pair directly.

The original baseline metrics remain unchanged so that the reranked system can be compared fairly against them.

In [52]:
from sentence_transformers import CrossEncoder


# Load a lightweight cross-encoder reranker
reranker_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(
    reranker_model_name
)


# Use a wider first-stage candidate pool
rerank_candidate_k = 20

# Final evidence passed forward
rerank_top_k = 10


def hybrid_retrieve_with_reranker(
    question,
    document_name,
    candidate_k=rerank_candidate_k,
    top_k=rerank_top_k,
):
    """
    Retrieve candidates with semantic search + BM25 + RRF,
    then rerank the candidate pool using a cross-encoder.
    """

    # Retrieve the union of a wider semantic and BM25 candidate set
    candidates = hybrid_retrieve_for_evaluation(
        question=question,
        document_name=document_name,
        semantic_k=candidate_k,
        bm25_k=candidate_k,
        top_k=candidate_k * 2,
    ).copy()


    # Add the original chunk text needed by the reranker
    text_lookup = (
        chunks_df[
            chunks_df["document_name"] == document_name
        ]
        .set_index("chunk_id")["text"]
        .to_dict()
    )


    candidates["text"] = candidates[
        "chunk_id"
    ].map(text_lookup)


    # Create question-chunk pairs for cross-encoder scoring
    question_chunk_pairs = [
        [
            question,
            chunk_text,
        ]
        for chunk_text in candidates["text"]
    ]


    # Score each candidate directly against the question
    reranker_scores = reranker.predict(
        question_chunk_pairs
    )


    candidates["reranker_score"] = (
        reranker_scores
    )


    # Higher cross-encoder score means greater relevance
    reranked = (
        candidates
        .sort_values(
            "reranker_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )


    reranked["reranker_rank"] = (
        reranked.index + 1
    )


    return reranked.head(
        top_k
    )

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [53]:
eval_001 = gold_eval_df[
    gold_eval_df["question_id"] == "EVAL_001"
].iloc[0]


reranked_test = hybrid_retrieve_with_reranker(
    question=eval_001["question"],
    document_name=eval_001["document_name"],
)


print(
    reranked_test[
        [
            "reranker_rank",
            "chunk_id",
            "page_number",
            "semantic_rank",
            "bm25_rank",
            "rrf_score",
            "reranker_score",
        ]
    ].to_string(index=False)
)

 reranker_rank            chunk_id  page_number  semantic_rank  bm25_rank  rrf_score  reranker_score
             1 PROTO_001_P003_C001            3            3.0        2.0   0.032002        2.006048
             2 PROTO_001_P004_C003            4            2.0        6.0   0.031281        0.404925
             3 PROTO_001_P003_C003            3            6.0       10.0   0.029437        0.001262
             4 PROTO_001_P003_C002            3            1.0        3.0   0.032266       -0.126112
             5 PROTO_001_P002_C004            2            5.0        5.0   0.030769       -4.663536
             6 PROTO_001_P004_C001            4            4.0        4.0   0.031250       -5.143978
             7 PROTO_001_P001_C002            1           17.0        1.0   0.029380       -5.560157
             8 PROTO_001_P005_C003            5           10.0        8.0   0.028992       -8.051191
             9 PROTO_001_P002_C003            2            9.0       17.0   0.027480       

## 47. Evaluate the reranked retrieval pipeline

The first reranker smoke test did not move the correct evidence into the top five.

A single question is not enough to determine whether the reranker is useful overall, so we will evaluate it across all 30 supported held-out questions.

The baseline metrics will remain unchanged. We will compare Hit@1, Hit@3, Hit@5, and Hit@10 between the original hybrid retrieval pipeline and the reranked pipeline.

In [54]:
reranked_evaluation_rows = []


for _, row in supported_eval.iterrows():

    # Run the wider candidate retrieval followed by reranking
    retrieved_df = hybrid_retrieve_with_reranker(
        question=row["question"],
        document_name=row["document_name"],
        candidate_k=20,
        top_k=10,
    )


    # Parse all acceptable gold chunks
    gold_chunk_ids = [
        chunk_id.strip()
        for chunk_id in row["gold_chunk_ids"].split(";")
        if chunk_id.strip()
    ]


    # Keep reranked chunk IDs in order
    retrieved_chunk_ids = (
        retrieved_df["chunk_id"]
        .tolist()
    )


    # Find ranks of acceptable gold chunks
    gold_ranks = [
        retrieved_chunk_ids.index(chunk_id) + 1
        for chunk_id in gold_chunk_ids
        if chunk_id in retrieved_chunk_ids
    ]


    best_gold_rank = (
        min(gold_ranks)
        if gold_ranks
        else None
    )


    reranked_evaluation_rows.append(
        {
            "question_id": row["question_id"],
            "document_name": row["document_name"],
            "question_type": row["question_type"],
            "best_gold_rank": best_gold_rank,
            "hit_at_1": (
                best_gold_rank is not None
                and best_gold_rank <= 1
            ),
            "hit_at_3": (
                best_gold_rank is not None
                and best_gold_rank <= 3
            ),
            "hit_at_5": (
                best_gold_rank is not None
                and best_gold_rank <= 5
            ),
            "hit_at_10": (
                best_gold_rank is not None
                and best_gold_rank <= 10
            ),
        }
    )


reranked_eval_df = pd.DataFrame(
    reranked_evaluation_rows
)


reranked_metrics = {
    "Hit@1": reranked_eval_df["hit_at_1"].mean(),
    "Hit@3": reranked_eval_df["hit_at_3"].mean(),
    "Hit@5": reranked_eval_df["hit_at_5"].mean(),
    "Hit@10": reranked_eval_df["hit_at_10"].mean(),
}


comparison_rows = []


for metric in [
    "Hit@1",
    "Hit@3",
    "Hit@5",
    "Hit@10",
]:

    comparison_rows.append(
        {
            "metric": metric,
            "baseline": retrieval_metrics[metric],
            "reranked": reranked_metrics[metric],
            "difference": (
                reranked_metrics[metric]
                - retrieval_metrics[metric]
            ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


print("Baseline vs reranked retrieval")
print()

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "baseline": "{:.3f}".format,
            "reranked": "{:.3f}".format,
            "difference": "{:+.3f}".format,
        },
    )
)


print("\nReranker Top-5 failures:")


reranker_failures = reranked_eval_df[
    ~reranked_eval_df["hit_at_5"]
]


print(
    reranker_failures[
        [
            "question_id",
            "document_name",
            "question_type",
            "best_gold_rank",
        ]
    ].to_string(index=False)
)

Baseline vs reranked retrieval

metric baseline reranked difference
 Hit@1    0.600    0.700     +0.100
 Hit@3    0.733    0.867     +0.133
 Hit@5    0.800    0.933     +0.133
Hit@10    0.933    0.967     +0.033

Reranker Top-5 failures:
question_id document_name   question_type  best_gold_rank
   EVAL_001       CERTAIN straightforward             7.0
   EVAL_028        LISTEN       difficult             NaN


## 48. Inspect the remaining retrieval failures

The reranked pipeline improved Hit@5 from 80.0% to 93.3%, leaving only two supported questions without acceptable gold evidence in the top five.

We will inspect these remaining failures without changing the retrieval configuration. This helps distinguish between candidate-retrieval and reranking errors and provides concrete examples for the final failure analysis.

In [55]:
remaining_failure_ids = [
    "EVAL_001",
    "EVAL_028",
]


for question_id in remaining_failure_ids:

    row = gold_eval_df[
        gold_eval_df["question_id"] == question_id
    ].iloc[0]

    gold_ids = [
        chunk_id.strip()
        for chunk_id in row["gold_chunk_ids"].split(";")
        if chunk_id.strip()
    ]


    # Keep the same frozen reranker configuration,
    # but return the whole candidate pool for diagnosis.
    diagnostic_results = hybrid_retrieve_with_reranker(
        question=row["question"],
        document_name=row["document_name"],
        candidate_k=20,
        top_k=40,
    )


    gold_matches = diagnostic_results[
        diagnostic_results["chunk_id"].isin(gold_ids)
    ]


    print()
    print("=" * 70)
    print(question_id)
    print(row["question"])
    print("Gold chunks:", gold_ids)


    if gold_matches.empty:

        print("Gold evidence was absent from the candidate pool.")

    else:

        print("\nGold evidence ranking:")

        print(
            gold_matches[
                [
                    "chunk_id",
                    "semantic_rank",
                    "bm25_rank",
                    "hybrid_rank",
                    "reranker_rank",
                    "reranker_score",
                ]
            ].to_string(index=False)
        )


    print("\nReranked Top 5:")

    print(
        diagnostic_results[
            [
                "reranker_rank",
                "chunk_id",
                "page_number",
                "reranker_score",
            ]
        ]
        .head(5)
        .to_string(index=False)
    )


    print("\nGold evidence text:")

    for gold_id in gold_ids:

        gold_text = chunks_df.loc[
            chunks_df["chunk_id"] == gold_id,
            "text",
        ]

        if not gold_text.empty:

            print()
            print(gold_id)
            print(
                gold_text.iloc[0][:1200]
                .replace("\n", " ")
            )


EVAL_001
How long were participants in the CERTAIN study followed after the first counselling session?
Gold chunks: ['PROTO_001_P001_C002']

Gold evidence ranking:
           chunk_id  semantic_rank  bm25_rank  hybrid_rank  reranker_rank  reranker_score
PROTO_001_P001_C002           17.0        1.0            7              7       -5.560157

Reranked Top 5:
 reranker_rank            chunk_id  page_number  reranker_score
             1 PROTO_001_P003_C001            3        2.006048
             2 PROTO_001_P004_C003            4        0.404925
             3 PROTO_001_P003_C003            3        0.001262
             4 PROTO_001_P003_C002            3       -0.126112
             5 PROTO_001_P002_C004            2       -4.663536

Gold evidence text:

PROTO_001_P001_C002
phfi.   org Protocol © Author(s) (or their  employer(s)) 2022. Re-  use  permitted under CC BY .  Published by BMJ. ABSTRACT Introduction Despite widespread use of smokeless  tobacco products by people within the

## 49. Classify the remaining LISTEN retrieval failure

EVAL_001 was confirmed as a reranking failure: the correct evidence entered the candidate pool but remained below the final top five.

We will now inspect only the ranking information for EVAL_028 so that the final remaining retrieval failure can be classified without producing another large notebook output.

In [56]:
# Get the remaining LISTEN failure
eval_028 = gold_eval_df[
    gold_eval_df["question_id"] == "EVAL_028"
].iloc[0]


gold_ids = [
    chunk_id.strip()
    for chunk_id in eval_028["gold_chunk_ids"].split(";")
    if chunk_id.strip()
]


# Return the full candidate pool
diagnostic_results = hybrid_retrieve_with_reranker(
    question=eval_028["question"],
    document_name=eval_028["document_name"],
    candidate_k=20,
    top_k=40,
)


# Check whether the gold evidence entered the candidate pool
gold_matches = diagnostic_results[
    diagnostic_results["chunk_id"].isin(gold_ids)
]


print("Question:", eval_028["question"])
print("Gold chunks:", gold_ids)
print()


if gold_matches.empty:

    print("Classification: CANDIDATE_RECALL_MISS")
    print("Gold evidence was absent from the candidate pool.")

else:

    print("Classification: RERANKING_MISS")
    print()

    print(
        gold_matches[
            [
                "chunk_id",
                "semantic_rank",
                "bm25_rank",
                "hybrid_rank",
                "reranker_rank",
                "reranker_score",
            ]
        ].to_string(index=False)
    )

Question: What methods were planned for the LISTEN process evaluation?
Gold chunks: ['PROTO_004_P003_C001']

Classification: RERANKING_MISS

           chunk_id  semantic_rank  bm25_rank  hybrid_rank  reranker_rank  reranker_score
PROTO_004_P003_C001           20.0        9.0           11             15       -2.082098


## 50. Build the RAG pipeline for full evaluation

Retrieval evaluation is now complete.

The improved retrieval pipeline will use a wider hybrid candidate pool followed by cross-encoder reranking, with the five highest-ranked chunks passed to the language model.

The generation rules remain evidence-only: factual claims must be supported by retrieved chunks, citations must use exact chunk IDs, and unsupported questions should return `Insufficient Evidence`.

No further retrieval tuning will be performed before the full RAG evaluation.

In [57]:
# Keep the same generation behaviour used on Day 4
generation_model_name = "gpt-5.6-terra"
abstention_message = "Insufficient Evidence"

# Five evidence chunks will be passed to the generator
generation_top_k = 5


def format_reranked_evidence(retrieved_df):
    """
    Convert reranked chunks into a clear evidence block for generation.
    """

    evidence_sections = []

    for source_number, (_, row) in enumerate(
        retrieved_df.iterrows(),
        start=1,
    ):

        evidence_sections.append(
            f"""Source {source_number}
Chunk ID: {row['chunk_id']}
Protocol: {row['document_name']}
Page: {row['page_number']}
Evidence:
{row['text']}"""
        )

    return "\n\n".join(evidence_sections)


def build_evaluation_prompt(question, context):
    """
    Build the strict evidence-only prompt used during evaluation.
    """

    return f"""
You are answering a question about a clinical trial protocol.

Use only the evidence provided below.

Rules:
1. Do not use outside knowledge.
2. Do not invent, assume, or infer missing information.
3. If the evidence does not support the answer, respond exactly:
   {abstention_message}
4. Every factual claim in a supported answer must have a citation.
5. Place each citation directly after the claim it supports.
6. Citations must use the exact format [CHUNK_ID].
7. Preserve differences between groups, time points, outcomes, and study procedures.
8. Do not present ambiguous table or flowchart extraction as certain.
9. Prefer clearly stated evidence over uncertain layout-derived interpretation.
10. Keep the answer concise.

Question:
{question}

Evidence:
{context}
""".strip()


def validate_evaluation_citations(answer, retrieved_chunk_ids):
    """
    Check whether citations use chunk IDs from the supplied evidence.
    """

    cited_chunk_ids = re.findall(
        r"\[(PROTO_\d+_P\d+_C\d+)\]",
        answer,
    )

    invalid_citations = [
        chunk_id
        for chunk_id in cited_chunk_ids
        if chunk_id not in retrieved_chunk_ids
    ]

    has_citations = len(cited_chunk_ids) > 0

    return {
        "cited_chunk_ids": cited_chunk_ids,
        "citation_count": len(cited_chunk_ids),
        "has_citations": has_citations,
        "invalid_citations": invalid_citations,
        "all_citations_valid": len(invalid_citations) == 0,
    }


def ask_protocol_question_reranked(
    question,
    document_name,
):
    """
    Run the complete evaluation pipeline:
    hybrid retrieval → reranking → Top 5 evidence →
    grounded generation → citation validation.
    """

    # Retrieve and rerank evidence
    retrieved_df = hybrid_retrieve_with_reranker(
        question=question,
        document_name=document_name,
        candidate_k=20,
        top_k=generation_top_k,
    )


    # Format retrieved evidence for the model
    context = format_reranked_evidence(
        retrieved_df
    )


    # Build the strict grounded prompt
    prompt = build_evaluation_prompt(
        question,
        context,
    )


    # Generate the answer
    response = openai_client.responses.create(
        model=generation_model_name,
        input=prompt,
    )

    answer = response.output_text.strip()


    retrieved_chunk_ids = (
        retrieved_df["chunk_id"]
        .tolist()
    )


    # Abstention is valid without citations
    if answer == abstention_message:

        citation_validation = {
            "cited_chunk_ids": [],
            "citation_count": 0,
            "has_citations": False,
            "invalid_citations": [],
            "all_citations_valid": True,
        }

    else:

        citation_validation = validate_evaluation_citations(
            answer,
            retrieved_chunk_ids,
        )


    return {
        "question": question,
        "document_name": document_name,
        "answer": answer,
        "citation_validation": citation_validation,
        "evidence": retrieved_df,
    }

In [58]:
supported_smoke = ask_protocol_question_reranked(
    question=gold_eval_df.loc[
        gold_eval_df["question_id"] == "EVAL_018",
        "question",
    ].iloc[0],
    document_name="INTEGRA",
)


unsupported_smoke = ask_protocol_question_reranked(
    question=gold_eval_df.loc[
        gold_eval_df["question_id"] == "EVAL_040",
        "question",
    ].iloc[0],
    document_name="THP_TA",
)


print("SUPPORTED TEST")
print(supported_smoke["answer"])
print(supported_smoke["citation_validation"])

print("\nUNSUPPORTED TEST")
print(unsupported_smoke["answer"])
print(unsupported_smoke["citation_validation"])

SUPPORTED TEST
The coaching training programme for primary-care professionals was 7 hours long. [PROTO_003_P005_C001]
{'cited_chunk_ids': ['PROTO_003_P005_C001'], 'citation_count': 1, 'has_citations': True, 'invalid_citations': [], 'all_citations_valid': True}

UNSUPPORTED TEST
Insufficient Evidence
{'cited_chunk_ids': [], 'citation_count': 0, 'has_citations': False, 'invalid_citations': [], 'all_citations_valid': True}


## 51. Run the full RAG evaluation

The supported and unsupported smoke tests both behaved correctly.

We will now run the complete reranked RAG pipeline across all 40 frozen evaluation questions.

For each question, we will record the generated answer, retrieved evidence, citation behaviour, and whether the system abstained. Results are saved incrementally so that completed evaluations are preserved if the run is interrupted.

Answer correctness and claim-level grounding will be evaluated separately after the full run.

In [59]:
evaluation_run_rows = []


# Save raw predictions here as the evaluation progresses
evaluation_predictions_path = (
    project_root
    / "data"
    / "metadata"
    / "rag_evaluation_predictions.csv"
)


for run_number, (_, row) in enumerate(
    gold_eval_df.iterrows(),
    start=1,
):

    question_id = row["question_id"]

    print(
        f"[{run_number:02d}/40] "
        f"{question_id} | {row['document_name']}"
    )


    try:

        # Run the complete reranked RAG pipeline
        result = ask_protocol_question_reranked(
            question=row["question"],
            document_name=row["document_name"],
        )


        answer = result["answer"]

        citation_validation = (
            result["citation_validation"]
        )

        retrieved_df = result["evidence"]


        # Keep the retrieved Top 5 IDs for later failure analysis
        retrieved_chunk_ids = (
            retrieved_df["chunk_id"]
            .tolist()
        )


        # Did the model abstain?
        returned_abstention = (
            answer == abstention_message
        )


        # Should it have abstained?
        expected_abstention = (
            row["expected_support"]
            == "unsupported"
        )


        # Record structural evaluation information
        evaluation_run_rows.append(
            {
                "question_id": question_id,
                "document_id": row["document_id"],
                "document_name": row["document_name"],
                "question_type": row["question_type"],
                "expected_support": row["expected_support"],
                "question": row["question"],
                "gold_chunk_ids": row["gold_chunk_ids"],
                "gold_answer_notes": row["gold_answer_notes"],
                "answer": answer,
                "returned_abstention": returned_abstention,
                "expected_abstention": expected_abstention,
                "citation_count": citation_validation[
                    "citation_count"
                ],
                "has_citations": citation_validation[
                    "has_citations"
                ],
                "all_citations_valid": citation_validation[
                    "all_citations_valid"
                ],
                "invalid_citations": ";".join(
                    citation_validation[
                        "invalid_citations"
                    ]
                ),
                "cited_chunk_ids": ";".join(
                    citation_validation[
                        "cited_chunk_ids"
                    ]
                ),
                "retrieved_top_5": ";".join(
                    retrieved_chunk_ids
                ),
                "run_error": "",
            }
        )


    except Exception as error:

        # Preserve the failure instead of losing the whole run
        evaluation_run_rows.append(
            {
                "question_id": question_id,
                "document_id": row["document_id"],
                "document_name": row["document_name"],
                "question_type": row["question_type"],
                "expected_support": row["expected_support"],
                "question": row["question"],
                "gold_chunk_ids": row["gold_chunk_ids"],
                "gold_answer_notes": row["gold_answer_notes"],
                "answer": "",
                "returned_abstention": False,
                "expected_abstention": (
                    row["expected_support"]
                    == "unsupported"
                ),
                "citation_count": 0,
                "has_citations": False,
                "all_citations_valid": False,
                "invalid_citations": "",
                "cited_chunk_ids": "",
                "retrieved_top_5": "",
                "run_error": str(error),
            }
        )

        print(
            f"  ERROR: {error}"
        )


    # Save after every question
    evaluation_predictions_df = pd.DataFrame(
        evaluation_run_rows
    )

    evaluation_predictions_df.to_csv(
        evaluation_predictions_path,
        index=False,
    )


print()
print("Full RAG evaluation run complete.")
print(
    "Completed rows:",
    len(evaluation_predictions_df),
)

print(
    "Run errors:",
    (
        evaluation_predictions_df["run_error"]
        .fillna("")
        .ne("")
        .sum()
    ),
)

print()
print("Saved to:")
print(evaluation_predictions_path)

[01/40] EVAL_001 | CERTAIN
[02/40] EVAL_002 | CERTAIN
[03/40] EVAL_003 | CERTAIN
[04/40] EVAL_004 | CERTAIN
[05/40] EVAL_005 | CERTAIN
[06/40] EVAL_006 | CERTAIN
[07/40] EVAL_007 | CERTAIN
[08/40] EVAL_008 | CERTAIN
[09/40] EVAL_009 | CARE_STROKE
[10/40] EVAL_010 | CARE_STROKE
[11/40] EVAL_011 | CARE_STROKE
[12/40] EVAL_012 | CARE_STROKE
[13/40] EVAL_013 | CARE_STROKE
[14/40] EVAL_014 | CARE_STROKE
[15/40] EVAL_015 | CARE_STROKE
[16/40] EVAL_016 | CARE_STROKE
[17/40] EVAL_017 | INTEGRA
[18/40] EVAL_018 | INTEGRA
[19/40] EVAL_019 | INTEGRA
[20/40] EVAL_020 | INTEGRA
[21/40] EVAL_021 | INTEGRA
[22/40] EVAL_022 | INTEGRA
[23/40] EVAL_023 | INTEGRA
[24/40] EVAL_024 | INTEGRA
[25/40] EVAL_025 | LISTEN
[26/40] EVAL_026 | LISTEN
[27/40] EVAL_027 | LISTEN
[28/40] EVAL_028 | LISTEN
[29/40] EVAL_029 | LISTEN
[30/40] EVAL_030 | LISTEN
[31/40] EVAL_031 | LISTEN
[32/40] EVAL_032 | LISTEN
[33/40] EVAL_033 | THP_TA
[34/40] EVAL_034 | THP_TA
[35/40] EVAL_035 | THP_TA
[36/40] EVAL_036 | THP_TA
[37/40] 

In [60]:
print("Rows saved:", len(evaluation_predictions_df))

print(
    evaluation_predictions_df[
        ["question_id", "document_name"]
    ].tail(5).to_string(index=False)
)

Rows saved: 40
question_id document_name
   EVAL_036        THP_TA
   EVAL_037        THP_TA
   EVAL_038        THP_TA
   EVAL_039        THP_TA
   EVAL_040        THP_TA


## 52. Measure abstention and citation reliability

The complete RAG evaluation finished without execution errors.

We will first measure behaviours that can be checked automatically: whether unsupported questions correctly returned `Insufficient Evidence`, whether supported questions were incorrectly rejected, and whether generated answers used valid citations.

These metrics do not yet measure whether every factual claim in an answer is correct. Answer correctness and claim grounding will be evaluated separately.

In [61]:
# Confirm that the complete evaluation run was saved
assert len(evaluation_predictions_df) == 40, (
    f"Expected 40 evaluation rows, found {len(evaluation_predictions_df)}."
)

assert (
    evaluation_predictions_df["run_error"]
    .fillna("")
    .eq("")
    .all()
), "Some evaluation questions produced run errors."


# Separate supported and unsupported questions
supported_predictions = evaluation_predictions_df[
    evaluation_predictions_df["expected_support"] == "supported"
].copy()

unsupported_predictions = evaluation_predictions_df[
    evaluation_predictions_df["expected_support"] == "unsupported"
].copy()


# Unsupported questions should return Insufficient Evidence
unsupported_abstention_rate = (
    unsupported_predictions["returned_abstention"].mean()
)

failed_to_abstain = (
    ~unsupported_predictions["returned_abstention"]
).sum()


# Supported questions should normally produce an answer
over_abstention_count = (
    supported_predictions["returned_abstention"]
).sum()

supported_answer_rate = (
    ~supported_predictions["returned_abstention"]
).mean()


# Citation metrics only apply to supported answers that were actually generated
generated_supported_answers = supported_predictions[
    ~supported_predictions["returned_abstention"]
].copy()


citation_coverage = (
    generated_supported_answers["has_citations"].mean()
)

citation_validity = (
    generated_supported_answers["all_citations_valid"].mean()
)

invalid_citation_answers = (
    ~generated_supported_answers["all_citations_valid"]
).sum()


print("End-to-end structural reliability metrics")
print()

print(
    f"Unsupported abstention rate: "
    f"{unsupported_abstention_rate:.3f} "
    f"({unsupported_predictions['returned_abstention'].sum()}/"
    f"{len(unsupported_predictions)})"
)

print(
    f"Failed to abstain: "
    f"{failed_to_abstain}"
)

print(
    f"Supported answer rate: "
    f"{supported_answer_rate:.3f} "
    f"({(~supported_predictions['returned_abstention']).sum()}/"
    f"{len(supported_predictions)})"
)

print(
    f"Over-abstentions: "
    f"{over_abstention_count}"
)

print(
    f"Citation coverage on generated supported answers: "
    f"{citation_coverage:.3f} "
    f"({generated_supported_answers['has_citations'].sum()}/"
    f"{len(generated_supported_answers)})"
)

print(
    f"Citation validity on generated supported answers: "
    f"{citation_validity:.3f} "
    f"({generated_supported_answers['all_citations_valid'].sum()}/"
    f"{len(generated_supported_answers)})"
)

print(
    f"Answers containing invalid citations: "
    f"{invalid_citation_answers}"
)

End-to-end structural reliability metrics

Unsupported abstention rate: 1.000 (10/10)
Failed to abstain: 0
Supported answer rate: 1.000 (30/30)
Over-abstentions: 0
Citation coverage on generated supported answers: 1.000 (30/30)
Citation validity on generated supported answers: 1.000 (30/30)
Answers containing invalid citations: 0


## 53. Measure citation alignment with gold evidence

The structural reliability checks passed for all 40 evaluation questions.

We will now perform a stronger grounding check on the 30 supported questions. For each generated answer, we will compare its cited chunk IDs with the manually verified gold evidence recorded before evaluation.

This is stronger than citation-format validation because it checks whether the model cited evidence that we had independently identified as answer-bearing.

It still does not replace manual answer-correctness review, which will be performed next.

In [62]:
gold_alignment_rows = []


for _, row in supported_predictions.iterrows():

    # Parse the manually verified acceptable gold chunks
    gold_ids = {
        chunk_id.strip()
        for chunk_id in str(row["gold_chunk_ids"]).split(";")
        if chunk_id.strip()
    }


    # Parse chunks cited by the generated answer
    cited_ids = {
        chunk_id.strip()
        for chunk_id in str(row["cited_chunk_ids"]).split(";")
        if chunk_id.strip()
    }


    # Parse the final Top 5 retrieved evidence
    retrieved_ids = {
        chunk_id.strip()
        for chunk_id in str(row["retrieved_top_5"]).split(";")
        if chunk_id.strip()
    }


    # Did retrieval put acceptable gold evidence in the final context?
    gold_in_top_5 = bool(
        gold_ids & retrieved_ids
    )


    # Did the generated answer cite at least one acceptable gold chunk?
    cited_gold_evidence = bool(
        gold_ids & cited_ids
    )


    gold_alignment_rows.append(
        {
            "question_id": row["question_id"],
            "document_name": row["document_name"],
            "question_type": row["question_type"],
            "gold_in_top_5": gold_in_top_5,
            "cited_gold_evidence": cited_gold_evidence,
            "gold_chunk_ids": row["gold_chunk_ids"],
            "cited_chunk_ids": row["cited_chunk_ids"],
        }
    )


gold_alignment_df = pd.DataFrame(
    gold_alignment_rows
)


# Calculate aggregate metrics
gold_context_rate = (
    gold_alignment_df["gold_in_top_5"].mean()
)

gold_citation_rate = (
    gold_alignment_df["cited_gold_evidence"].mean()
)


print("Gold-evidence alignment")
print()

print(
    f"Gold evidence present in final Top 5: "
    f"{gold_context_rate:.3f} "
    f"({gold_alignment_df['gold_in_top_5'].sum()}/30)"
)

print(
    f"Generated answer cited gold evidence: "
    f"{gold_citation_rate:.3f} "
    f"({gold_alignment_df['cited_gold_evidence'].sum()}/30)"
)


# Show only questions that did not cite manually verified gold evidence
citation_alignment_failures = gold_alignment_df[
    ~gold_alignment_df["cited_gold_evidence"]
].copy()


print()
print(
    "Answers without a citation to annotated gold evidence:",
    len(citation_alignment_failures),
)


if len(citation_alignment_failures) > 0:

    print()

    print(
        citation_alignment_failures[
            [
                "question_id",
                "document_name",
                "question_type",
                "gold_in_top_5",
                "gold_chunk_ids",
                "cited_chunk_ids",
            ]
        ].to_string(index=False)
    )

Gold-evidence alignment

Gold evidence present in final Top 5: 0.933 (28/30)
Generated answer cited gold evidence: 0.867 (26/30)

Answers without a citation to annotated gold evidence: 4

question_id document_name   question_type  gold_in_top_5      gold_chunk_ids                                                                                                                                                 cited_chunk_ids
   EVAL_001       CERTAIN straightforward          False PROTO_001_P001_C002                                                                                                                                             PROTO_001_P003_C002
   EVAL_027        LISTEN       difficult           True PROTO_004_P001_C002                                                                                                                         PROTO_004_P005_C001;PROTO_004_P005_C001
   EVAL_028        LISTEN       difficult          False PROTO_004_P003_C001 PROTO_004_P007_C001;PROT

## 54. Inspect answers that did not cite annotated gold evidence

Four supported answers did not cite one of the chunks originally annotated as acceptable gold evidence.

This does not automatically mean the answers are incorrect. An overlapping or alternative chunk may independently support the same fact.

We will compare the generated answer, manually verified gold answer, gold evidence, and cited evidence before assigning a failure type.

In [63]:
# Inspect only the four questions requiring manual grounding review
review_question_ids = [
    "EVAL_001",
    "EVAL_027",
    "EVAL_028",
    "EVAL_035",
]


def show_chunk_excerpt(chunk_id, max_chars=700):
    """
    Return a compact excerpt for one chunk.
    """

    match = chunks_df[
        chunks_df["chunk_id"] == chunk_id
    ]

    if match.empty:
        return "[Chunk not found]"

    text = (
        match.iloc[0]["text"]
        .replace("\n", " ")
    )

    return text[:max_chars]


for question_id in review_question_ids:

    row = evaluation_predictions_df[
        evaluation_predictions_df["question_id"] == question_id
    ].iloc[0]


    gold_ids = list(dict.fromkeys([
        chunk_id.strip()
        for chunk_id in str(row["gold_chunk_ids"]).split(";")
        if chunk_id.strip()
    ]))

    cited_ids = list(dict.fromkeys([
        chunk_id.strip()
        for chunk_id in str(row["cited_chunk_ids"]).split(";")
        if chunk_id.strip()
    ]))


    print()
    print("=" * 80)
    print(question_id, "|", row["document_name"])

    print("\nQUESTION")
    print(row["question"])

    print("\nGOLD ANSWER")
    print(row["gold_answer_notes"])

    print("\nGENERATED ANSWER")
    print(row["answer"])

    print("\nGOLD EVIDENCE")

    for chunk_id in gold_ids:
        print(f"\n{chunk_id}")
        print(show_chunk_excerpt(chunk_id))

    print("\nCITED EVIDENCE")

    for chunk_id in cited_ids:
        print(f"\n{chunk_id}")
        print(show_chunk_excerpt(chunk_id))


EVAL_001 | CERTAIN

QUESTION
How long were participants in the CERTAIN study followed after the first counselling session?

GOLD ANSWER
All participants were followed for 3 months from the first counselling session.

GENERATED ANSWER
Participants were followed for 3 months after the first counselling session. [PROTO_001_P003_C002]

GOLD EVIDENCE

PROTO_001_P001_C002
phfi.   org Protocol © Author(s) (or their  employer(s)) 2022. Re-  use  permitted under CC BY .  Published by BMJ. ABSTRACT Introduction Despite widespread use of smokeless  tobacco products by people within the Indian subcontinent,   there is little awareness among Indians of its health  hazards when compared with smoked tobacco. We  hypothesise that mobile phone counselling will be  feasible and effective for smokeless tobacco cessation  intervention in India. This paper presents the protocol of  the development and conduct of an exploratory trial before  progression to a full randomised controlled trial. Methods and an

## 55. Complete the manual grounding review

One apparent citation mismatch was already confirmed to use valid alternative evidence.

We will now inspect only the relevant wording inside the cited chunks for the remaining exceptions. This allows us to distinguish genuine grounding failures from cases where our original gold annotation did not include every valid supporting chunk.

In [64]:
review_terms = {
    "EVAL_001": [
        "3 months",
        "followed",
    ],
    "EVAL_027": [
        "six",
        "one-to-one",
        "digital resource",
        "paper",
    ],
    "EVAL_028": [
        "implementation",
        "interviews",
        "focus groups",
        "process evaluation",
    ],
}


for question_id, search_terms in review_terms.items():

    row = evaluation_predictions_df[
        evaluation_predictions_df["question_id"] == question_id
    ].iloc[0]

    cited_ids = list(dict.fromkeys([
        chunk_id.strip()
        for chunk_id in str(row["cited_chunk_ids"]).split(";")
        if chunk_id.strip()
    ]))

    print()
    print("=" * 70)
    print(question_id)

    print("\nQUESTION:")
    print(row["question"])

    print("\nGOLD ANSWER:")
    print(row["gold_answer_notes"])

    print("\nGENERATED ANSWER:")
    print(row["answer"])


    for chunk_id in cited_ids:

        chunk_match = chunks_df[
            chunks_df["chunk_id"] == chunk_id
        ]

        if chunk_match.empty:
            continue

        text = (
            chunk_match.iloc[0]["text"]
            .replace("\n", " ")
        )

        print(f"\nCITED CHUNK: {chunk_id}")

        found_any = False

        for term in search_terms:

            position = text.lower().find(
                term.lower()
            )

            if position >= 0:

                found_any = True

                start = max(
                    0,
                    position - 180,
                )

                end = min(
                    len(text),
                    position + len(term) + 350,
                )

                print(
                    f"\nAround '{term}':"
                )

                print(
                    text[start:end]
                )

        if not found_any:
            print(
                "None of the expected supporting terms "
                "were found in this cited chunk."
            )


EVAL_001

QUESTION:
How long were participants in the CERTAIN study followed after the first counselling session?

GOLD ANSWER:
All participants were followed for 3 months from the first counselling session.

GENERATED ANSWER:
Participants were followed for 3 months after the first counselling session. [PROTO_001_P003_C002]

CITED CHUNK: PROTO_001_P003_C002

Around '3 months':
figur e illustrating the phases of CERTAIN trial and data collection time points Study period Pre-   enrolment Enrolment Allocation Post allocation Close-   out Time point 0 0 0 0 3 months After 3 months Enrolment    Eligibility scr eening X    Informed consent X    Randomisation to tr eatment allocation X    Interventions    Routine car e X    T en-   minute face-   to-   face counselling X    Mobile message-  based counselling X    Assessments    Demographic X    Baseline assessment X    Mid-   line assessment—qualitative  assessm

EVAL_027

QUESTION:
What intervention support was offered to participants rando

## 56. Complete the remaining grounding checks

Two apparent gold-citation mismatches have already been confirmed as valid alternative evidence.

We will now check the remaining LISTEN cases using compact term-level evidence checks instead of printing full chunks.

In [65]:
remaining_grounding_checks = {
    "EVAL_027": [
        "six",
        "one-to-one",
        "digital resource",
        "paper-based",
    ],
    "EVAL_028": [
        "implementation scales",
        "interviews",
        "focus groups",
    ],
}


check_rows = []


for question_id, expected_terms in remaining_grounding_checks.items():

    row = evaluation_predictions_df[
        evaluation_predictions_df["question_id"] == question_id
    ].iloc[0]


    cited_ids = list(dict.fromkeys([
        chunk_id.strip()
        for chunk_id in str(row["cited_chunk_ids"]).split(";")
        if chunk_id.strip()
    ]))


    # Combine all cited evidence text for this answer
    cited_text_parts = []

    for chunk_id in cited_ids:

        match = chunks_df[
            chunks_df["chunk_id"] == chunk_id
        ]

        if not match.empty:
            cited_text_parts.append(
                match.iloc[0]["text"]
            )


    cited_text = " ".join(
        cited_text_parts
    ).lower()


    term_results = {
        term: term.lower() in cited_text
        for term in expected_terms
    }


    check_rows.append(
        {
            "question_id": question_id,
            "generated_answer": row["answer"],
            "cited_chunk_ids": ";".join(cited_ids),
            **term_results,
        }
    )


remaining_grounding_df = pd.DataFrame(
    check_rows
)


print(
    remaining_grounding_df.to_string(
        index=False
    )
)

question_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

## 57. Verify alternative supporting evidence

The remaining apparent citation mismatches use different wording from the original gold annotations.

Before calculating the final grounding result, we will verify that the additional factual details in the generated LISTEN answers are explicitly supported by the chunks they cited.

In [66]:
alternative_evidence_checks = {
    "EVAL_027": [
        "six",
        "one-to-one",
        "10 weeks",
        "1 hour",
        "video",
        "telephone",
        "print",
        "web",
        "3-month",
    ],
    "EVAL_028": [
        "AIM",
        "IAM",
        "FIM",
        "interviews",
        "focus groups",
        "acceptability",
        "feasibility",
        "appropriateness",
    ],
}


for question_id, terms in alternative_evidence_checks.items():

    row = evaluation_predictions_df[
        evaluation_predictions_df["question_id"] == question_id
    ].iloc[0]

    cited_ids = list(dict.fromkeys([
        chunk_id.strip()
        for chunk_id in str(row["cited_chunk_ids"]).split(";")
        if chunk_id.strip()
    ]))

    cited_text = " ".join(
        chunks_df.loc[
            chunks_df["chunk_id"].isin(cited_ids),
            "text",
        ]
    ).lower()

    print()
    print(question_id)

    for term in terms:
        print(
            f"{term}:",
            term.lower() in cited_text,
        )


EVAL_027
six: True
one-to-one: True
10 weeks: False
1 hour: False
video: True
telephone: True
print: True
web: True
3-month: True

EVAL_028
AIM: True
IAM: True
FIM: True
interviews: True
focus groups: True
acceptability: True
feasibility: True
appropriateness: True


## 58. Check the remaining EVAL_027 details

Most of the generated LISTEN intervention answer is explicitly supported by its cited evidence.

Two details were not found using exact phrase matching: the 10-week delivery period and sessions lasting up to 1 hour. We will check common wording variants before deciding whether these are unsupported additions.

In [67]:
# Get the cited evidence for EVAL_027
eval_027_row = evaluation_predictions_df[
    evaluation_predictions_df["question_id"] == "EVAL_027"
].iloc[0]


eval_027_cited_ids = list(dict.fromkeys([
    chunk_id.strip()
    for chunk_id in str(
        eval_027_row["cited_chunk_ids"]
    ).split(";")
    if chunk_id.strip()
]))


eval_027_cited_text = " ".join(
    chunks_df.loc[
        chunks_df["chunk_id"].isin(
            eval_027_cited_ids
        ),
        "text",
    ]
).lower()


# Check alternative ways the two details may appear
variant_checks = {
    "10 weeks": [
        "10 weeks",
        "10-week",
        "ten weeks",
        "ten-week",
    ],
    "1 hour": [
        "1 hour",
        "one hour",
        "60 min",
        "60 minutes",
    ],
}


for fact, variants in variant_checks.items():

    matching_variants = [
        variant
        for variant in variants
        if variant in eval_027_cited_text
    ]

    print(
        fact,
        "→",
        matching_variants
        if matching_variants
        else "NOT FOUND",
    )

10 weeks → ['10-week']
1 hour → NOT FOUND


## 59. Evaluate supported-answer correctness

The previous checks measured retrieval, abstention, citation validity, and claim grounding.

We will now compare each of the 30 supported generated answers with the manually verified gold answer.

An LLM will be used only as a first-pass evaluator to classify answers as `PASS`, `PARTIAL`, or `FAIL`. Because automated judging can make mistakes, all non-passing cases will be manually reviewed before final correctness metrics are reported.

In [68]:
correctness_rows = []


def judge_answer_correctness(question, gold_answer, generated_answer):
    """
    Compare a generated answer with the manually verified gold answer.

    The judge checks factual correctness and completeness, not wording.
    """

    judge_prompt = f"""
You are evaluating the factual correctness of an answer to a clinical-trial protocol question.

Use the GOLD ANSWER as the reference.

Classify the GENERATED ANSWER as exactly one of:

PASS
- Correctly answers the question.
- Contains all important information required by the gold answer.
- Different wording is acceptable.
- Extra information is acceptable only if it does not contradict the gold answer.

PARTIAL
- Core answer is partly correct but an important required detail is missing,
  unclear, or slightly inaccurate.

FAIL
- Answer is materially incorrect, contradicts the gold answer,
  or fails to answer the question.

Do not judge citation formatting here.
Do not use outside knowledge.

QUESTION:
{question}

GOLD ANSWER:
{gold_answer}

GENERATED ANSWER:
{generated_answer}

Return exactly two lines:

LABEL: PASS|PARTIAL|FAIL
REASON: brief explanation
""".strip()


    response = openai_client.responses.create(
        model=generation_model_name,
        input=judge_prompt,
    )

    judge_text = response.output_text.strip()


    # Parse the two expected lines
    label = ""

    reason = ""

    for line in judge_text.splitlines():

        if line.startswith("LABEL:"):
            label = line.replace(
                "LABEL:",
                "",
                1,
            ).strip().upper()

        elif line.startswith("REASON:"):
            reason = line.replace(
                "REASON:",
                "",
                1,
            ).strip()


    if label not in {
        "PASS",
        "PARTIAL",
        "FAIL",
    }:
        label = "REVIEW"


    return label, reason


for run_number, (_, row) in enumerate(
    supported_predictions.iterrows(),
    start=1,
):

    print(
        f"[{run_number:02d}/30] "
        f"{row['question_id']}"
    )


    label, reason = judge_answer_correctness(
        question=row["question"],
        gold_answer=row["gold_answer_notes"],
        generated_answer=row["answer"],
    )


    correctness_rows.append(
        {
            "question_id": row["question_id"],
            "document_name": row["document_name"],
            "question_type": row["question_type"],
            "correctness_label": label,
            "judge_reason": reason,
        }
    )


correctness_df = pd.DataFrame(
    correctness_rows
)


print()
print("First-pass correctness results:")
print(
    correctness_df[
        "correctness_label"
    ]
    .value_counts()
    .to_string()
)


print("\nCases requiring manual review:")

manual_review_df = correctness_df[
    correctness_df["correctness_label"]
    != "PASS"
].copy()


if manual_review_df.empty:

    print("None")

else:

    print(
        manual_review_df[
            [
                "question_id",
                "document_name",
                "question_type",
                "correctness_label",
                "judge_reason",
            ]
        ].to_string(index=False)
    )

[01/30] EVAL_001
[02/30] EVAL_002
[03/30] EVAL_003
[04/30] EVAL_004
[05/30] EVAL_005
[06/30] EVAL_006
[07/30] EVAL_009
[08/30] EVAL_010
[09/30] EVAL_011
[10/30] EVAL_012
[11/30] EVAL_013
[12/30] EVAL_014
[13/30] EVAL_017
[14/30] EVAL_018
[15/30] EVAL_019
[16/30] EVAL_020
[17/30] EVAL_021
[18/30] EVAL_022
[19/30] EVAL_025
[20/30] EVAL_026
[21/30] EVAL_027
[22/30] EVAL_028
[23/30] EVAL_029
[24/30] EVAL_030
[25/30] EVAL_033
[26/30] EVAL_034
[27/30] EVAL_035
[28/30] EVAL_036
[29/30] EVAL_037
[30/30] EVAL_038

First-pass correctness results:
correctness_label
PASS    30

Cases requiring manual review:
None


## 60. Manually audit representative answers

The automated correctness judge classified all 30 supported answers as correct.

Because an LLM judge can introduce evaluation bias, this result will not be treated as final accuracy on its own.

We will manually inspect a small representative sample covering difficult, confusable, and structure-heavy questions. The generated answer will be compared directly with the manually verified gold answer.

In [69]:
# Select representative questions from the harder evaluation categories
manual_audit_ids = [
    "EVAL_005",  # CERTAIN - confusable
    "EVAL_014",  # CARE_STROKE - structure-heavy
    "EVAL_021",  # INTEGRA - confusable
    "EVAL_030",  # LISTEN - structure-heavy
    "EVAL_035",  # THP_TA - difficult
    "EVAL_037",  # THP_TA - confusable
]


audit_rows = []


for question_id in manual_audit_ids:

    row = evaluation_predictions_df[
        evaluation_predictions_df["question_id"] == question_id
    ].iloc[0]

    audit_rows.append(
        {
            "question_id": question_id,
            "question_type": row["question_type"],
            "question": row["question"],
            "gold_answer": row["gold_answer_notes"],
            "generated_answer": row["answer"],
        }
    )


manual_audit_df = pd.DataFrame(
    audit_rows
)


for _, row in manual_audit_df.iterrows():

    print()
    print("=" * 70)
    print(
        row["question_id"],
        "|",
        row["question_type"],
    )

    print("\nQUESTION")
    print(row["question"])

    print("\nGOLD ANSWER")
    print(row["gold_answer"])

    print("\nGENERATED ANSWER")
    print(row["generated_answer"])


EVAL_005 | confusable

QUESTION
Which CERTAIN study arms received the short Ask-and-Advice routine-care component?

GOLD ANSWER
The 1-to-2-minute routine-care component based on Ask and Advice was delivered to both the intervention and control arms.

GENERATED ANSWER
Both the intervention arm and the control arm received the short routine-care “Ask” and “Advice” component (component 1). [PROTO_001_P003_C002]

EVAL_014 | structure_heavy

QUESTION
What adverse events were specifically expected during the CARE_STROKE trial?

GOLD ANSWER
Expected adverse events included death from vascular causes, hospitalisation for post-stroke complications, and occurrence of a secondary stroke.

GENERATED ANSWER
Expected adverse events were: death from vascular causes (e.g., myocardial infarction or recurrent stroke); hospitalisation for post-stroke complications (including infections, brain oedema, seizures, deep vein thrombosis, urinary tract infections, pressure sores, and shoulder subluxation, disl

In [70]:
for question_id in [
    "EVAL_021",
    "EVAL_030",
    "EVAL_035",
]:
    row = evaluation_predictions_df[
        evaluation_predictions_df["question_id"] == question_id
    ].iloc[0]

    print()
    print(question_id)
    print("GOLD:", row["gold_answer_notes"])
    print("ANSWER:", row["answer"])


EVAL_021
GOLD: IG-1 included the monographic consultation, while IG-2 received the otherwise similar personalised intervention without the monographic consultation.
ANSWER: IG-1 included a specific monographic (specialized) consultation, whereas IG-2 received the same personalized intervention without that consultation. [PROTO_003_P005_C002]

EVAL_030
GOLD: Process-evaluation questionnaires were completed only by participants in the LISTEN intervention group.
ANSWER: Only participants in the LISTEN intervention group completed the process-evaluation questionnaires. [PROTO_004_P004_C002]

EVAL_035
GOLD: The primary outcome was remission from major depressive episode at 3 months postnatal, measured using the SCID.
ANSWER: The primary outcome was remission from major depressive episode, assessed at 3 months postnatal using the Structured Clinical Interview for DSM-IV Disorders (SCID) Major Depressive Episode module. [PROTO_005_P007_C003]


## 61. Final evaluation results and failure analysis

The complete evaluation combined retrieval testing, answer generation, abstention checks, citation validation, gold-evidence comparison, and manual review.

The reranker substantially improved retrieval ranking, while the generation pipeline correctly answered all supported questions according to the automated evaluator and a representative manual audit.

The evaluation also exposed an important reliability distinction: a citation can be structurally valid while an individual factual detail is still unsupported by that citation.

The remaining observed failures are therefore recorded explicitly rather than hidden behind aggregate metrics.

In [72]:
# Record the manually reviewed grounding exceptions
grounding_review = pd.DataFrame(
    [
        {
            "question_id": "EVAL_001",
            "classification": "CORRECT_ALTERNATIVE_EVIDENCE",
            "notes": (
                "Generated answer was correct and the cited alternative "
                "chunk independently contained the 3-month follow-up evidence."
            ),
        },
        {
            "question_id": "EVAL_027",
            "classification": "UNSUPPORTED_EXTRA_DETAIL",
            "notes": (
                "Core LISTEN intervention answer was correct, but the claim "
                "that sessions lasted up to 1 hour was not found in the cited evidence."
            ),
        },
        {
            "question_id": "EVAL_028",
            "classification": "CORRECT_ALTERNATIVE_EVIDENCE",
            "notes": (
                "Alternative cited chunks supported implementation scales, "
                "interviews, focus groups, and related process-evaluation details."
            ),
        },
        {
            "question_id": "EVAL_035",
            "classification": "CORRECT_ALTERNATIVE_EVIDENCE",
            "notes": (
                "The cited page-7 chunk independently stated the primary "
                "outcome, 3-month timing, and SCID measurement."
            ),
        },
    ]
)


# Build the final retrieval comparison from the actual measured metrics
final_retrieval_results = pd.DataFrame(
    {
        "Metric": [
            "Hit@1",
            "Hit@3",
            "Hit@5",
            "Hit@10",
        ],
        "Baseline": [
            retrieval_metrics["Hit@1"],
            retrieval_metrics["Hit@3"],
            retrieval_metrics["Hit@5"],
            retrieval_metrics["Hit@10"],
        ],
        "Reranked": [
            reranked_metrics["Hit@1"],
            reranked_metrics["Hit@3"],
            reranked_metrics["Hit@5"],
            reranked_metrics["Hit@10"],
        ],
    }
)


# Report the absolute percentage-point improvement
final_retrieval_results["Percentage-point gain"] = (
    final_retrieval_results["Reranked"]
    - final_retrieval_results["Baseline"]
)


print("FINAL RETRIEVAL RESULTS")
print()

print(
    final_retrieval_results.to_string(
        index=False,
        formatters={
            "Baseline": "{:.1%}".format,
            "Reranked": "{:.1%}".format,
            "Percentage-point gain": "{:+.1%}".format,
        },
    )
)

FINAL RETRIEVAL RESULTS

Metric Baseline Reranked Percentage-point gain
 Hit@1    60.0%    70.0%                +10.0%
 Hit@3    73.3%    86.7%                +13.3%
 Hit@5    80.0%    93.3%                +13.3%
Hit@10    93.3%    96.7%                 +3.3%


## Day 5 findings

Day 5 evaluated the protocol QA system using a manually verified 40-question benchmark covering all five protocols.

The evaluation set contained 30 supported questions and 10 deliberately unsupported questions. Supported questions included straightforward, difficult, confusable, and structure-heavy cases. The gold answers and acceptable evidence chunks were defined before the RAG system was evaluated.

### Retrieval

The original hybrid retriever combined OpenAI semantic search and BM25 using Reciprocal Rank Fusion.

Its baseline retrieval performance was:

- Hit@1: 60.0% (18/30)
- Hit@3: 73.3% (22/30)
- Hit@5: 80.0% (24/30)
- Hit@10: 93.3% (28/30)

Failure analysis showed that most missed Top-5 cases were ranking problems rather than complete evidence-recall failures.

A wider candidate pool followed by a cross-encoder reranker improved retrieval to:

- Hit@1: 70.0% (21/30)
- Hit@3: 86.7% (26/30)
- Hit@5: 93.3% (28/30)
- Hit@10: 96.7% (29/30)

The largest practical improvement was Hit@5, which increased from 80.0% to 93.3%.

### End-to-end RAG behaviour

The complete reranked RAG pipeline was evaluated on all 40 questions.

- Supported questions answered: 30/30
- Unsupported questions correctly returned `Insufficient Evidence`: 10/10
- Over-abstentions: 0
- Failed abstentions: 0
- Supported answers containing citations: 30/30
- Answers using only retrieved chunk IDs: 30/30
- Annotated gold evidence present in the final Top 5: 28/30
- Answers citing an originally annotated gold chunk: 26/30

The four citation-to-gold mismatches were manually reviewed. Three used alternative protocol chunks that independently supported the answer.

One answer, EVAL_027, contained a confirmed claim-level grounding issue: the core LISTEN intervention answer was correct, but the model added that sessions lasted "up to 1 hour", which was not supported by the chunk cited for that claim.

This demonstrated an important reliability limitation:

**A structurally valid citation does not guarantee that every factual claim is supported by that citation.**

### Answer correctness

An automated first-pass evaluator classified all 30 supported answers as correct.

Because LLM-based judging can introduce bias, six representative difficult, confusable, and structure-heavy answers were also manually reviewed. All six matched the manually verified gold answers.

The automated 30/30 result is therefore treated as an evaluation signal supported by manual spot-checking rather than as proof of universal 100% accuracy.

### Main engineering conclusions

The evaluation showed that retrieval ranking was a larger weakness than basic candidate recall. A reranker was therefore justified by measured failures rather than added by assumption.

Hybrid lexical and semantic retrieval remained useful because BM25 recovered evidence that semantic retrieval sometimes ranked poorly.

The evaluation also showed why citation validity, evidence retrieval, answer correctness, abstention, and claim grounding should be measured separately rather than collapsed into a single accuracy number.

These findings will guide the production implementation and final reliability safeguards.